In [ ]:
# ==================================================================================================
# XGBOOST NOTEBOOK — CELL 1
#
# STARTUP + FROZEN PROTOCOL VERIFICATION
#
# NO TEST PATH IS DEFINED.
# ==================================================================================================

from google.colab import drive
drive.mount(
    "/content/drive"
)

from pathlib import Path
import json
import hashlib
import subprocess
import sys
import shutil
import gc

import numpy as np
import pandas as pd

from packaging.version import Version


print("=" * 120)
print("XGBOOST W10 TEMPORAL-SUMMARY FORECASTER")
print("CELL 1 — STARTUP / PROTOCOL VERIFICATION")
print("=" * 120)


# ==================================================================================================
# PACKAGE CHECK
# ==================================================================================================

try:

    import xgboost as xgb

except ImportError:

    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "xgboost",
        ]
    )

    import xgboost as xgb


try:

    import optuna

except ImportError:

    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "optuna",
        ]
    )

    import optuna


print(
    "XGBoost version:",
    xgb.__version__
)

print(
    "Optuna version :",
    optuna.__version__
)


# ==================================================================================================
# GPU DETECTION
# ==================================================================================================

GPU_AVAILABLE = (
    shutil.which(
        "nvidia-smi"
    )
    is not None
)


print(
    "GPU available:",
    GPU_AVAILABLE
)


if GPU_AVAILABLE:

    try:

        gpu_name = subprocess.check_output(
            [
                "nvidia-smi",
                "--query-gpu=name",
                "--format=csv,noheader"
            ],
            text=True
        ).strip()

        print(
            "GPU:",
            gpu_name
        )

    except Exception:

        pass


if not GPU_AVAILABLE:

    print(
        "\nNOTE: XGBoost will still work on CPU."
    )

    print(
        "For faster HPO, enable a T4 GPU before Cell 3."
    )


# ==================================================================================================
# PATHS
# ==================================================================================================

ROOT = Path(
    "/content/drive/MyDrive/network-attack-forecasting/Synthetic_NAF_v1"
)

P12_0 = (
    ROOT
    / "model_ready"
    / "12_0"
)

P12_3 = (
    ROOT
    / "model_ready"
    / "12_3"
)

MODEL_ROOT = (
    ROOT
    / "models"
    / "XGBOOST_W10_TEMPORAL_SUMMARY"
)

MODEL_ROOT.mkdir(
    parents=True,
    exist_ok=True
)


# --------------------------------------------------------------------------------------------------
# Phase 12.3
# --------------------------------------------------------------------------------------------------

READY_FILE = (
    P12_3
    / "12_3_READY_FOR_MODELING.json"
)

LOCK_FILE = (
    P12_3
    / "12_3_PROTOCOL_LOCK.json"
)

LOCK_HASH_FILE = (
    P12_3
    / "12_3_PROTOCOL_LOCK_SHA256.txt"
)


# --------------------------------------------------------------------------------------------------
# TRAIN
# --------------------------------------------------------------------------------------------------

TRAIN_X_FILE = (
    P12_0
    / "W10"
    / "train"
    / "X.npy"
)

TRAIN_Y_FILE = (
    P12_0
    / "W10"
    / "train"
    / "y_binary.npy"
)

TRAIN_ELIGIBLE_FILE = (
    P12_0
    / "W10"
    / "train"
    / "eligible.npy"
)

TRAIN_INDEX_FILE = (
    P12_0
    / "W10"
    / "train"
    / "index.parquet"
)


# --------------------------------------------------------------------------------------------------
# VALIDATION
# --------------------------------------------------------------------------------------------------

VAL_X_FILE = (
    P12_0
    / "W10"
    / "validation"
    / "X.npy"
)

VAL_Y_FILE = (
    P12_0
    / "W10"
    / "validation"
    / "y_binary.npy"
)

VAL_ELIGIBLE_FILE = (
    P12_0
    / "W10"
    / "validation"
    / "eligible.npy"
)

VAL_INDEX_FILE = (
    P12_0
    / "W10"
    / "validation"
    / "index.parquet"
)


# ==================================================================================================
# IMPORTANT
#
# THERE IS DELIBERATELY NO:
#
# TEST_X_FILE
# TEST_Y_FILE
# TEST_INDEX_FILE
#
# ==================================================================================================


# ==================================================================================================
# HELPERS
# ==================================================================================================

def require_file(path):

    if not path.is_file():

        raise FileNotFoundError(
            f"Required artifact missing:\n{path}"
        )


def read_json(path):

    require_file(
        path
    )

    with open(
        path,
        "r",
        encoding="utf-8"
    ) as f:

        return json.load(
            f
        )


def sha256_file(path):

    h = hashlib.sha256()

    with open(
        path,
        "rb"
    ) as f:

        while True:

            chunk = f.read(
                1024 * 1024
            )

            if not chunk:

                break

            h.update(
                chunk
            )

    return h.hexdigest()


def write_json(
    obj,
    path
):

    path.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    with open(
        path,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            obj,
            f,
            indent=4
        )


# ==================================================================================================
# VERIFY INPUTS
# ==================================================================================================

required_files = [
    READY_FILE,
    LOCK_FILE,
    LOCK_HASH_FILE,

    TRAIN_X_FILE,
    TRAIN_Y_FILE,
    TRAIN_ELIGIBLE_FILE,
    TRAIN_INDEX_FILE,

    VAL_X_FILE,
    VAL_Y_FILE,
    VAL_ELIGIBLE_FILE,
    VAL_INDEX_FILE,
]


for path in required_files:

    require_file(
        path
    )


ready = read_json(
    READY_FILE
)


if ready.get(
    "phase_12_3_status"
) != "PASS":

    raise RuntimeError(
        "Phase 12.3 is not PASS."
    )


if ready.get(
    "training_authorized"
) is not True:

    raise RuntimeError(
        "Training is not authorized."
    )


if ready.get(
    "test_evaluation_authorized"
) is not False:

    raise RuntimeError(
        "Unexpected TEST authorization."
    )


current_hash = sha256_file(
    LOCK_FILE
)

frozen_hash = (
    LOCK_HASH_FILE
    .read_text(
        encoding="utf-8"
    )
    .strip()
)


if current_hash != frozen_hash:

    raise RuntimeError(
        "Phase 12.3 protocol hash mismatch."
    )


print(
    "\n✓ Phase 12.3 PASS"
)

print(
    "✓ TRAIN / VALIDATION modeling authorized"
)

print(
    "✓ TEST remains embargoed"
)

print(
    "✓ Protocol hash verified"
)


# ==================================================================================================
# BASIC SHAPE CHECK
# ==================================================================================================

X_train = np.load(
    TRAIN_X_FILE,
    mmap_mode="r"
)

X_val = np.load(
    VAL_X_FILE,
    mmap_mode="r"
)


print(
    "\nRaw W10 TRAIN shape:",
    X_train.shape
)

print(
    "Raw W10 VALIDATION shape:",
    X_val.shape
)


if X_train.shape != (
    59837,
    10,
    14
):

    raise RuntimeError(
        f"Unexpected TRAIN shape: {X_train.shape}"
    )


if X_val.shape != (
    12661,
    10,
    14
):

    raise RuntimeError(
        f"Unexpected VALIDATION shape: {X_val.shape}"
    )


# ==================================================================================================
# EXPERIMENT CONTRACT
# ==================================================================================================

write_json(
    {
        "Model":
            "XGBOOST",

        "History_Window":
            10,

        "Primary_Horizons":
            [
                5,
                10
            ],

        "Input":
            "RAW_PHASE_12_0_W10",

        "Temporal_Features":
            [
                "LAST",
                "MEAN",
                "STD",
                "MIN",
                "MAX",
                "DELTA_LAST_MINUS_FIRST",
                "LINEAR_SLOPE"
            ],

        "Per_Minute_Transform":
            "log1p",

        "Base_Features":
            14,

        "Temporal_Statistics":
            7,

        "Final_Feature_Count":
            98,

        "HPO":
            "3-fold episode-level TRAIN CV",

        "Headline_Metric":
            "Average Precision",

        "Official_Validation_Used_For_HPO":
            False,

        "TEST_Path_Defined":
            False,

        "TEST_Loaded":
            False,

        "TEST_Used":
            False,

        "Phase_12_3_SHA256":
            current_hash,
    },

    MODEL_ROOT
    / "XGBOOST_EXPERIMENT_CONTRACT.json"
)


del X_train
del X_val

gc.collect()


print("\n" + "=" * 120)
print("✓ CELL 1 COMPLETE")
print("=" * 120)

print(
    "Experiment folder:"
)

print(
    MODEL_ROOT
)

print(
    "\nTEST path defined: NO ✓"
)

print(
    "\nRun CELL 2 next."
)

In [ ]:
# ==================================================================================================
# XGBOOST NOTEBOOK — CELL 2
#
# BUILD 98 TEMPORAL-SUMMARY FEATURES
#
# Deterministic + label-free.
# No fitting.
# No target information.
# No TEST.
# ==================================================================================================

from pathlib import Path
import json
import hashlib
import gc

import numpy as np
import pandas as pd


print("=" * 120)
print("CELL 2 — BUILD W10 TEMPORAL-SUMMARY FEATURES")
print("=" * 120)


# ==================================================================================================
# PATHS — REDECLARED DELIBERATELY
# ==================================================================================================

ROOT = Path(
    "/content/drive/MyDrive/network-attack-forecasting/Synthetic_NAF_v1"
)

P12_0 = (
    ROOT
    / "model_ready"
    / "12_0"
)

MODEL_ROOT = (
    ROOT
    / "models"
    / "XGBOOST_W10_TEMPORAL_SUMMARY"
)

FEATURE_DIR = (
    MODEL_ROOT
    / "temporal_features"
)

FEATURE_DIR.mkdir(
    parents=True,
    exist_ok=True
)


TRAIN_X_FILE = (
    P12_0
    / "W10"
    / "train"
    / "X.npy"
)

VAL_X_FILE = (
    P12_0
    / "W10"
    / "validation"
    / "X.npy"
)


TRAIN_SUMMARY_FILE = (
    FEATURE_DIR
    / "X_train_log1p_temporal98.npy"
)

VAL_SUMMARY_FILE = (
    FEATURE_DIR
    / "X_validation_log1p_temporal98.npy"
)

FEATURE_NAMES_FILE = (
    FEATURE_DIR
    / "feature_names.json"
)


# ==================================================================================================
# FROZEN 14 FEATURES
# ==================================================================================================

BASE_FEATURES = [
    "Flow_Count",
    "Flow_Duration_Mean",
    "Flow_Duration_Std",
    "Flow_Duration_Max",
    "Forward_Packets_Sum",
    "Backward_Packets_Sum",
    "Total_Packets",
    "Forward_Bytes_Sum",
    "Backward_Bytes_Sum",
    "Total_Bytes",
    "Packets_Per_Second",
    "Bytes_Per_Second",
    "Forward_Backward_Packet_Ratio",
    "Forward_Backward_Byte_Ratio",
]


STATISTICS = [
    "LAST",
    "MEAN",
    "STD",
    "MIN",
    "MAX",
    "DELTA",
    "SLOPE",
]


# ==================================================================================================
# TEMPORAL FEATURE FUNCTION
# ==================================================================================================

def build_temporal_summary(
    X_raw
):

    X_raw = np.asarray(
        X_raw,
        dtype=np.float32
    )


    if X_raw.ndim != 3:

        raise RuntimeError(
            f"Expected 3D tensor, got {X_raw.shape}"
        )


    if X_raw.shape[
        1:
    ] != (
        10,
        14
    ):

        raise RuntimeError(
            f"Expected [N,10,14], got {X_raw.shape}"
        )


    if not np.isfinite(
        X_raw
    ).all():

        raise RuntimeError(
            "Raw X contains non-finite values."
        )


    if np.any(
        X_raw
        <
        0
    ):

        raise RuntimeError(
            "Raw X contains negative values; log1p blocked."
        )


    # ----------------------------------------------------------------------------------------------
    # Per-minute log1p.
    #
    # Useful for network traffic magnitudes where counts/rates/bytes
    # may be highly right-skewed.
    # ----------------------------------------------------------------------------------------------

    X = np.log1p(
        X_raw
    ).astype(
        np.float32
    )


    last = X[
        :,
        -1,
        :
    ]


    mean = X.mean(
        axis=1
    )


    std = X.std(
        axis=1,
        ddof=0
    )


    minimum = X.min(
        axis=1
    )


    maximum = X.max(
        axis=1
    )


    delta = (
        X[
            :,
            -1,
            :
        ]
        -
        X[
            :,
            0,
            :
        ]
    )


    # ----------------------------------------------------------------------------------------------
    # OLS temporal slope over positions 0...9.
    #
    # slope = covariance(time, feature) / variance(time)
    # ----------------------------------------------------------------------------------------------

    t = np.arange(
        X.shape[
            1
        ],
        dtype=np.float32
    )


    t_centered = (
        t
        -
        t.mean()
    )


    denominator = np.sum(
        t_centered ** 2
    )


    slope = (

        np.sum(
            X
            *
            t_centered[
                None,
                :,
                None
            ],
            axis=1
        )

        /

        denominator
    )


    summary = np.concatenate(
        [
            last,
            mean,
            std,
            minimum,
            maximum,
            delta,
            slope,
        ],
        axis=1
    ).astype(
        np.float32
    )


    if summary.shape[
        1
    ] != 98:

        raise RuntimeError(
            f"Expected 98 features, got {summary.shape}"
        )


    if not np.isfinite(
        summary
    ).all():

        raise RuntimeError(
            "Temporal summary contains non-finite values."
        )


    return summary


# ==================================================================================================
# FEATURE NAMES
# ==================================================================================================

FEATURE_NAMES = []


for statistic in STATISTICS:

    for feature in BASE_FEATURES:

        FEATURE_NAMES.append(
            f"{statistic}__{feature}"
        )


if len(
    FEATURE_NAMES
) != 98:

    raise RuntimeError(
        "Feature-name count is not 98."
    )


# ==================================================================================================
# BUILD TRAIN
# ==================================================================================================

X_train_raw = np.load(
    TRAIN_X_FILE,
    mmap_mode="r"
)


print(
    "\nBuilding TRAIN temporal summaries..."
)


X_train_summary = build_temporal_summary(
    X_train_raw
)


np.save(
    TRAIN_SUMMARY_FILE,
    X_train_summary
)


print(
    "✓ TRAIN:",
    X_train_summary.shape
)


# ==================================================================================================
# BUILD VALIDATION
# ==================================================================================================

X_val_raw = np.load(
    VAL_X_FILE,
    mmap_mode="r"
)


print(
    "\nBuilding VALIDATION temporal summaries..."
)


X_val_summary = build_temporal_summary(
    X_val_raw
)


np.save(
    VAL_SUMMARY_FILE,
    X_val_summary
)


print(
    "✓ VALIDATION:",
    X_val_summary.shape
)


# ==================================================================================================
# SAVE FEATURE NAMES
# ==================================================================================================

with open(
    FEATURE_NAMES_FILE,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        FEATURE_NAMES,
        f,
        indent=4
    )


pd.DataFrame(
    {
        "Feature_Index":
            np.arange(
                98
            ),

        "Feature_Name":
            FEATURE_NAMES,
    }
).to_csv(
    FEATURE_DIR
    / "feature_names.csv",
    index=False
)


# ==================================================================================================
# AUDIT
# ==================================================================================================

print("\n" + "=" * 120)
print("FEATURE AUDIT")
print("=" * 120)


print(
    "TRAIN finite:",
    bool(
        np.isfinite(
            X_train_summary
        ).all()
    )
)

print(
    "VALIDATION finite:",
    bool(
        np.isfinite(
            X_val_summary
        ).all()
    )
)

print(
    "Feature count:",
    X_train_summary.shape[
        1
    ]
)

print(
    "Targets used to construct predictors: NO"
)

print(
    "Episode metadata used as predictors: NO"
)

print(
    "Family/source/regime used as predictors: NO"
)

print(
    "TEST used: NO"
)


# ==================================================================================================
# QUICK FEATURE PREVIEW
# ==================================================================================================

preview = pd.DataFrame(
    X_train_summary[
        :5
    ],
    columns=FEATURE_NAMES
)


print(
    "\nFirst 5 rows / first 12 features:"
)


print(
    preview.iloc[
        :,
        :12
    ].to_string(
        index=False
    )
)


del X_train_raw
del X_val_raw
del X_train_summary
del X_val_summary

gc.collect()


print("\n" + "=" * 120)
print("✓ CELL 2 COMPLETE")
print("=" * 120)

print(
    "98 deterministic temporal predictors built ✓"
)

print(
    "No learned preprocessing ✓"
)

print(
    "No target leakage ✓"
)

print(
    "No TEST ✓"
)

print(
    "\nRun CELL 3 next."
)

In [ ]:
# ==================================================================================================
# XGBOOST NOTEBOOK — CELL 3
#
# CLEAN EPISODE-LEVEL TRAIN CV + OPTUNA HPO
#
# OFFICIAL VALIDATION IS NOT LOADED.
# TEST IS NOT DEFINED.
#
# ==================================================================================================

from pathlib import Path
import json
import hashlib
import shutil
import gc
import subprocess
import sys

import numpy as np
import pandas as pd

import xgboost as xgb
import optuna

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    average_precision_score,
    precision_recall_curve,
    f1_score,
    matthews_corrcoef,
)


optuna.logging.set_verbosity(
    optuna.logging.WARNING
)


print("=" * 120)
print("CELL 3 — CLEAN TRAIN-ONLY EPISODE CV XGBOOST HPO")
print("=" * 120)


# ==================================================================================================
# PATHS — SELF-CONTAINED
# ==================================================================================================

ROOT = Path(
    "/content/drive/MyDrive/network-attack-forecasting/Synthetic_NAF_v1"
)

P12_0 = (
    ROOT
    / "model_ready"
    / "12_0"
)

MODEL_ROOT = (
    ROOT
    / "models"
    / "XGBOOST_W10_TEMPORAL_SUMMARY"
)

FEATURE_DIR = (
    MODEL_ROOT
    / "temporal_features"
)

HPO_DIR = (
    MODEL_ROOT
    / "hpo"
)

HPO_DIR.mkdir(
    parents=True,
    exist_ok=True
)


TRAIN_SUMMARY_FILE = (
    FEATURE_DIR
    / "X_train_log1p_temporal98.npy"
)

FEATURE_NAMES_FILE = (
    FEATURE_DIR
    / "feature_names.json"
)

TRAIN_Y_FILE = (
    P12_0
    / "W10"
    / "train"
    / "y_binary.npy"
)

TRAIN_ELIGIBLE_FILE = (
    P12_0
    / "W10"
    / "train"
    / "eligible.npy"
)

TRAIN_INDEX_FILE = (
    P12_0
    / "W10"
    / "train"
    / "index.parquet"
)


# ==================================================================================================
# CONSTANTS
# ==================================================================================================

HORIZONS = [
    5,
    10,
]

HORIZON_TO_COLUMN = {
    1: 0,
    5: 1,
    10: 2,
    15: 3,
    30: 4,
}

CV_FOLDS = 3

CV_SEED = 20261022

HPO_SEED = 20260940

TOTAL_TRIALS = 18

AP_TIE_TOLERANCE = 0.002


# ==================================================================================================
# DEVICE
# ==================================================================================================

GPU_AVAILABLE = (
    shutil.which(
        "nvidia-smi"
    )
    is not None
)


XGB_VERSION_MAJOR = int(
    xgb.__version__.split(
        "."
    )[0]
)


def device_params():

    if GPU_AVAILABLE:

        if XGB_VERSION_MAJOR >= 2:

            return {
                "tree_method":
                    "hist",

                "device":
                    "cuda",
            }

        else:

            return {
                "tree_method":
                    "gpu_hist",
            }

    return {
        "tree_method":
            "hist",

        "nthread":
            -1,
    }


print(
    "XGBoost version:",
    xgb.__version__
)

print(
    "GPU available:",
    GPU_AVAILABLE
)

print(
    "Device parameters:",
    device_params()
)


# ==================================================================================================
# LOAD TRAIN ONLY
# ==================================================================================================

required = [
    TRAIN_SUMMARY_FILE,
    FEATURE_NAMES_FILE,
    TRAIN_Y_FILE,
    TRAIN_ELIGIBLE_FILE,
    TRAIN_INDEX_FILE,
]


for path in required:

    if not path.is_file():

        raise FileNotFoundError(
            f"Missing:\n{path}"
        )


X_ALL = np.load(
    TRAIN_SUMMARY_FILE,
    mmap_mode="r"
)

Y_ALL = np.load(
    TRAIN_Y_FILE,
    allow_pickle=False
)

ELIGIBLE_ALL = np.load(
    TRAIN_ELIGIBLE_FILE,
    allow_pickle=False
)

INDEX_ALL = pd.read_parquet(
    TRAIN_INDEX_FILE
)


with open(
    FEATURE_NAMES_FILE,
    "r",
    encoding="utf-8"
) as f:

    FEATURE_NAMES = json.load(
        f
    )


if X_ALL.shape != (
    59837,
    98
):

    raise RuntimeError(
        f"Unexpected temporal-summary shape: {X_ALL.shape}"
    )


INDEX_ALL = INDEX_ALL.copy()

INDEX_ALL[
    "Episode_ID"
] = (
    INDEX_ALL[
        "Episode_ID"
    ]
    .astype(str)
)


EPISODE_IDS = INDEX_ALL[
    "Episode_ID"
].to_numpy()


# ==================================================================================================
# EPISODE STRATIFICATION
# ==================================================================================================

for column in [
    "Episode_Type",
    "Episode_Attack_Family",
    "Calibration_Source",
]:

    if column not in INDEX_ALL.columns:

        INDEX_ALL[
            column
        ] = "UNKNOWN"

    INDEX_ALL[
        column
    ] = (
        INDEX_ALL[
            column
        ]
        .astype("string")
        .fillna("NONE")
        .astype(str)
    )


episode_table = (

    INDEX_ALL[
        [
            "Episode_ID",
            "Episode_Type",
            "Episode_Attack_Family",
            "Calibration_Source",
        ]
    ]

    .groupby(
        "Episode_ID",
        as_index=False
    )

    .first()
)


episode_table[
    "Stratum"
] = (

    episode_table[
        "Episode_Type"
    ]

    +

    "|"

    +

    episode_table[
        "Episode_Attack_Family"
    ]

    +

    "|"

    +

    episode_table[
        "Calibration_Source"
    ]
)


if (
    episode_table[
        "Stratum"
    ]
    .value_counts()
    <
    CV_FOLDS
).any():

    episode_table[
        "Stratum"
    ] = (

        episode_table[
            "Episode_Type"
        ]

        +

        "|"

        +

        episode_table[
            "Episode_Attack_Family"
        ]
    )


print(
    "\nIndependent TRAIN episodes:",
    len(
        episode_table
    )
)


# ==================================================================================================
# CREATE FIXED CV SPLITS
# ==================================================================================================

skf = StratifiedKFold(
    n_splits=CV_FOLDS,
    shuffle=True,
    random_state=CV_SEED,
)


CV_SPLITS = []


for fold, (
    train_idx,
    dev_idx
) in enumerate(
    skf.split(
        episode_table[
            "Episode_ID"
        ],
        episode_table[
            "Stratum"
        ]
    ),
    start=1
):

    train_episodes = set(
        episode_table.iloc[
            train_idx
        ][
            "Episode_ID"
        ].astype(str)
    )


    dev_episodes = set(
        episode_table.iloc[
            dev_idx
        ][
            "Episode_ID"
        ].astype(str)
    )


    overlap = (
        train_episodes
        &
        dev_episodes
    )


    if overlap:

        raise RuntimeError(
            f"Fold {fold}: episode leakage."
        )


    CV_SPLITS.append(
        {
            "fold":
                fold,

            "train_episodes":
                train_episodes,

            "dev_episodes":
                dev_episodes,
        }
    )


    print(
        f"Fold {fold}: "
        f"TRAIN episodes={len(train_episodes)}, "
        f"DEV episodes={len(dev_episodes)}, "
        f"overlap=0"
    )


# ==================================================================================================
# SAVE CV ASSIGNMENTS
# ==================================================================================================

assignment_rows = []


for split in CV_SPLITS:

    for episode_id in split[
        "dev_episodes"
    ]:

        assignment_rows.append(
            {
                "Fold":
                    split[
                        "fold"
                    ],

                "Episode_ID":
                    episode_id,
            }
        )


assignments = pd.DataFrame(
    assignment_rows
)


assignments.to_csv(
    HPO_DIR
    / "CV_EPISODE_ASSIGNMENTS.csv",
    index=False
)


# ==================================================================================================
# CONTRACT HASH
#
# Prevents an old Optuna DB from silently belonging to another split/config.
# ==================================================================================================

contract_material = (
    assignments
    .sort_values(
        [
            "Fold",
            "Episode_ID"
        ]
    )
    .to_csv(
        index=False
    )
    +
    "\n"
    +
    "\n".join(
        FEATURE_NAMES
    )
)


CONTRACT_HASH = hashlib.sha256(
    contract_material.encode(
        "utf-8"
    )
).hexdigest()


print(
    "\nHPO contract hash:",
    CONTRACT_HASH
)


# ==================================================================================================
# METRIC HELPERS
# ==================================================================================================

def choose_f1_threshold(
    y,
    probability
):

    precision, recall, thresholds = (
        precision_recall_curve(
            y,
            probability
        )
    )


    if len(
        thresholds
    ) == 0:

        return 0.5


    precision = precision[
        :-1
    ]

    recall = recall[
        :-1
    ]


    denominator = (
        precision
        +
        recall
    )


    f1 = np.divide(
        2
        *
        precision
        *
        recall,
        denominator,
        out=np.zeros_like(
            denominator
        ),
        where=(
            denominator
            >
            0
        )
    )


    max_f1 = f1.max()


    candidates = np.flatnonzero(
        np.isclose(
            f1,
            max_f1,
            atol=1e-12,
            rtol=0
        )
    )


    candidate_df = pd.DataFrame(
        {
            "Index":
                candidates,

            "Precision":
                precision[
                    candidates
                ],

            "Threshold":
                thresholds[
                    candidates
                ],
        }
    )


    candidate_df = candidate_df.sort_values(
        [
            "Precision",
            "Threshold"
        ],
        ascending=[
            False,
            False
        ]
    )


    best_index = int(
        candidate_df.iloc[
            0
        ][
            "Index"
        ]
    )


    return float(
        thresholds[
            best_index
        ]
    )


# ==================================================================================================
# CONFIG
# ==================================================================================================

def resolve_trial_config(
    trial
):

    return {
        "max_depth":
            trial.suggest_int(
                "max_depth",
                3,
                8
            ),

        "min_child_weight":
            trial.suggest_float(
                "min_child_weight",
                1.0,
                20.0,
                log=True
            ),

        "learning_rate":
            trial.suggest_float(
                "learning_rate",
                0.015,
                0.15,
                log=True
            ),

        "subsample":
            trial.suggest_float(
                "subsample",
                0.65,
                1.0
            ),

        "colsample_bytree":
            trial.suggest_float(
                "colsample_bytree",
                0.55,
                1.0
            ),

        "gamma":
            trial.suggest_float(
                "gamma",
                0.0,
                3.0
            ),

        "reg_alpha":
            trial.suggest_float(
                "reg_alpha",
                1e-6,
                2.0,
                log=True
            ),

        "reg_lambda":
            trial.suggest_float(
                "reg_lambda",
                1e-2,
                20.0,
                log=True
            ),

        "scale_pos_weight_strategy":
            trial.suggest_categorical(
                "scale_pos_weight_strategy",
                [
                    "NONE",
                    "SQRT",
                    "FULL",
                ]
            ),
    }


def calculate_scale_pos_weight(
    y,
    strategy
):

    positives = float(
        np.sum(
            y
            ==
            1
        )
    )

    negatives = float(
        np.sum(
            y
            ==
            0
        )
    )


    ratio = (
        negatives
        /
        positives
    )


    if strategy == "NONE":

        return 1.0


    if strategy == "SQRT":

        return float(
            np.sqrt(
                ratio
            )
        )


    if strategy == "FULL":

        return float(
            ratio
        )


    raise ValueError(
        strategy
    )


# ==================================================================================================
# TRAIN ONE FOLD
# ==================================================================================================

def train_fold(
    *,
    H,
    fold_split,
    config,
    seed
):

    h_col = HORIZON_TO_COLUMN[
        H
    ]


    eligible = (
        ELIGIBLE_ALL[
            :,
            h_col
        ]
        ==
        1
    )


    train_mask = (

        eligible

        &

        np.isin(
            EPISODE_IDS,
            list(
                fold_split[
                    "train_episodes"
                ]
            )
        )
    )


    dev_mask = (

        eligible

        &

        np.isin(
            EPISODE_IDS,
            list(
                fold_split[
                    "dev_episodes"
                ]
            )
        )
    )


    if np.any(
        train_mask
        &
        dev_mask
    ):

        raise RuntimeError(
            "Fold row overlap."
        )


    X_train = np.asarray(
        X_ALL[
            train_mask
        ],
        dtype=np.float32
    )

    X_dev = np.asarray(
        X_ALL[
            dev_mask
        ],
        dtype=np.float32
    )


    y_train = np.asarray(
        Y_ALL[
            train_mask,
            h_col
        ],
        dtype=np.int8
    )

    y_dev = np.asarray(
        Y_ALL[
            dev_mask,
            h_col
        ],
        dtype=np.int8
    )


    scale_pos_weight = calculate_scale_pos_weight(
        y_train,
        config[
            "scale_pos_weight_strategy"
        ]
    )


    dtrain = xgb.DMatrix(
        X_train,
        label=y_train,
        feature_names=FEATURE_NAMES
    )

    ddev = xgb.DMatrix(
        X_dev,
        label=y_dev,
        feature_names=FEATURE_NAMES
    )


    params = {
        "objective":
            "binary:logistic",

        "eval_metric":
            "aucpr",

        "max_depth":
            config[
                "max_depth"
            ],

        "min_child_weight":
            config[
                "min_child_weight"
            ],

        "eta":
            config[
                "learning_rate"
            ],

        "subsample":
            config[
                "subsample"
            ],

        "colsample_bytree":
            config[
                "colsample_bytree"
            ],

        "gamma":
            config[
                "gamma"
            ],

        "alpha":
            config[
                "reg_alpha"
            ],

        "lambda":
            config[
                "reg_lambda"
            ],

        "scale_pos_weight":
            scale_pos_weight,

        "max_bin":
            256,

        "seed":
            seed,

        "verbosity":
            0,

        **device_params(),
    }


    model = xgb.train(
        params=params,
        dtrain=dtrain,
        num_boost_round=2500,
        evals=[
            (
                ddev,
                "dev"
            )
        ],
        early_stopping_rounds=100,
        verbose_eval=False,
    )


    best_iteration = int(
        model.best_iteration
    )


    probability = model.predict(
        ddev,
        iteration_range=(
            0,
            best_iteration
            +
            1
        )
    )


    ap = float(
        average_precision_score(
            y_dev,
            probability
        )
    )


    threshold = choose_f1_threshold(
        y_dev,
        probability
    )


    prediction = (
        probability
        >=
        threshold
    ).astype(
        np.int8
    )


    f1 = float(
        f1_score(
            y_dev,
            prediction,
            zero_division=0
        )
    )


    mcc = float(
        matthews_corrcoef(
            y_dev,
            prediction
        )
    )


    return {
        "AP":
            ap,

        "F1":
            f1,

        "MCC":
            mcc,

        "Best_Iteration":
            best_iteration,

        "Scale_Pos_Weight":
            scale_pos_weight,

        "Train_Rows":
            len(
                y_train
            ),

        "Dev_Rows":
            len(
                y_dev
            ),
    }


# ==================================================================================================
# HPO OBJECTIVE
# ==================================================================================================

def objective(
    trial,
    H
):

    config = resolve_trial_config(
        trial
    )


    fold_rows = []


    for fold_split in CV_SPLITS:

        result = train_fold(
            H=H,
            fold_split=fold_split,
            config=config,
            seed=(
                HPO_SEED
                +
                trial.number
                *
                10
                +
                fold_split[
                    "fold"
                ]
            ),
        )


        fold_rows.append(
            result
        )


        running_ap = float(
            np.mean(
                [
                    x[
                        "AP"
                    ]
                    for x
                    in fold_rows
                ]
            )
        )


        trial.report(
            running_ap,
            step=fold_split[
                "fold"
            ]
        )


        print(
            f"H{H} | "
            f"trial={trial.number:02d} | "
            f"fold={fold_split['fold']} | "
            f"AP={result['AP']:.6f} | "
            f"running={running_ap:.6f}"
        )


        if (
            fold_split[
                "fold"
            ]
            >=
            2

            and

            trial.should_prune()
        ):

            raise optuna.TrialPruned()


    aps = np.array(
        [
            x[
                "AP"
            ]
            for x
            in fold_rows
        ]
    )


    f1s = np.array(
        [
            x[
                "F1"
            ]
            for x
            in fold_rows
        ]
    )


    mccs = np.array(
        [
            x[
                "MCC"
            ]
            for x
            in fold_rows
        ]
    )


    iterations = np.array(
        [
            x[
                "Best_Iteration"
            ]
            for x
            in fold_rows
        ]
    )


    mean_ap = float(
        aps.mean()
    )


    trial.set_user_attr(
        "contract_hash",
        CONTRACT_HASH
    )

    trial.set_user_attr(
        "resolved_config",
        config
    )

    trial.set_user_attr(
        "fold_aps",
        aps.tolist()
    )

    trial.set_user_attr(
        "mean_ap",
        mean_ap
    )

    trial.set_user_attr(
        "std_ap",
        float(
            aps.std(
                ddof=1
            )
        )
    )

    trial.set_user_attr(
        "mean_f1",
        float(
            f1s.mean()
        )
    )

    trial.set_user_attr(
        "mean_mcc",
        float(
            mccs.mean()
        )
    )

    trial.set_user_attr(
        "mean_best_iteration",
        float(
            iterations.mean()
        )
    )


    print(
        f"H{H} TRIAL {trial.number} COMPLETE | "
        f"AP={mean_ap:.6f} ± "
        f"{aps.std(ddof=1):.6f}"
    )


    return mean_ap


# ==================================================================================================
# ANCHOR CONFIGURATIONS
# ==================================================================================================

ANCHORS = [
    {
        "max_depth":
            4,

        "min_child_weight":
            3.0,

        "learning_rate":
            0.05,

        "subsample":
            0.85,

        "colsample_bytree":
            0.85,

        "gamma":
            0.0,

        "reg_alpha":
            0.001,

        "reg_lambda":
            1.0,

        "scale_pos_weight_strategy":
            "NONE",
    },

    {
        "max_depth":
            5,

        "min_child_weight":
            5.0,

        "learning_rate":
            0.04,

        "subsample":
            0.85,

        "colsample_bytree":
            0.80,

        "gamma":
            0.25,

        "reg_alpha":
            0.01,

        "reg_lambda":
            2.0,

        "scale_pos_weight_strategy":
            "SQRT",
    },

    {
        "max_depth":
            6,

        "min_child_weight":
            5.0,

        "learning_rate":
            0.03,

        "subsample":
            0.80,

        "colsample_bytree":
            0.80,

        "gamma":
            0.5,

        "reg_alpha":
            0.05,

        "reg_lambda":
            3.0,

        "scale_pos_weight_strategy":
            "FULL",
    },
]


# ==================================================================================================
# RUN HPO
# ==================================================================================================

summary_rows = []


for H in HORIZONS:

    print("\n" + "#" * 120)

    print(
        f"XGBOOST HPO — H{H}"
    )

    print("#" * 120)


    study_name = (
        f"XGB_W10_H{H}_"
        +
        CONTRACT_HASH[
            :12
        ]
    )


    database = (
        HPO_DIR
        / "xgboost_optuna.sqlite3"
    )


    study = optuna.create_study(
        study_name=study_name,
        storage=(
            "sqlite:///"
            +
            str(
                database
            )
        ),
        direction="maximize",
        sampler=optuna.samplers.TPESampler(
            seed=HPO_SEED
            +
            H
        ),
        pruner=optuna.pruners.MedianPruner(
            n_startup_trials=5,
            n_warmup_steps=2
        ),
        load_if_exists=True,
    )


    # ----------------------------------------------------------------------------------------------
    # Safety: if study somehow contains a mismatched completed trial, stop.
    # ----------------------------------------------------------------------------------------------

    for existing_trial in study.trials:

        if (
            existing_trial.state
            ==
            optuna.trial.TrialState.COMPLETE
        ):

            trial_hash = existing_trial.user_attrs.get(
                "contract_hash"
            )


            if trial_hash != CONTRACT_HASH:

                raise RuntimeError(
                    "Existing Optuna study has wrong contract hash."
                )


    if len(
        study.trials
    ) == 0:

        for anchor in ANCHORS:

            study.enqueue_trial(
                anchor
            )


    finished = sum(
        trial.state
        in {
            optuna.trial.TrialState.COMPLETE,
            optuna.trial.TrialState.PRUNED,
            optuna.trial.TrialState.FAIL,
        }

        for trial
        in study.trials
    )


    remaining = max(
        0,
        TOTAL_TRIALS
        -
        finished
    )


    print(
        "Existing finished trials:",
        finished
    )

    print(
        "Remaining trials:",
        remaining
    )


    if remaining > 0:

        study.optimize(
            lambda trial:
                objective(
                    trial,
                    H
                ),
            n_trials=remaining,
            gc_after_trial=True,
        )


    complete_trials = [
        trial

        for trial
        in study.trials

        if trial.state
        ==
        optuna.trial.TrialState.COMPLETE
    ]


    if not complete_trials:

        raise RuntimeError(
            f"H{H}: no completed trials."
        )


    study.trials_dataframe().to_csv(
        HPO_DIR
        / f"H{H:02d}_OPTUNA_TRIALS.csv",
        index=False
    )


    # ----------------------------------------------------------------------------------------------
    # Selection:
    #
    # max mean CV AP.
    #
    # Anything within 0.002 AP is practically tied.
    # Among tied configurations:
    #   1. higher F1
    #   2. fewer boosting rounds
    #   3. lower trial number
    # ----------------------------------------------------------------------------------------------

    best_ap = max(
        float(
            trial.value
        )

        for trial
        in complete_trials
    )


    candidates = [
        trial

        for trial
        in complete_trials

        if (
            best_ap
            -
            float(
                trial.value
            )
        )
        <=
        AP_TIE_TOLERANCE
    ]


    candidates = sorted(
        candidates,
        key=lambda trial: (
            -float(
                trial.user_attrs[
                    "mean_f1"
                ]
            ),
            float(
                trial.user_attrs[
                    "mean_best_iteration"
                ]
            ),
            trial.number,
        )
    )


    selected = candidates[
        0
    ]


    selection = {
        "Horizon":
            H,

        "Trial":
            int(
                selected.number
            ),

        "Mean_CV_AP":
            float(
                selected.value
            ),

        "CV_AP_Std":
            float(
                selected.user_attrs[
                    "std_ap"
                ]
            ),

        "Mean_CV_F1":
            float(
                selected.user_attrs[
                    "mean_f1"
                ]
            ),

        "Mean_CV_MCC":
            float(
                selected.user_attrs[
                    "mean_mcc"
                ]
            ),

        "Mean_Best_Iteration":
            float(
                selected.user_attrs[
                    "mean_best_iteration"
                ]
            ),

        "Contract_Hash":
            CONTRACT_HASH,

        "Configuration":
            selected.user_attrs[
                "resolved_config"
            ],
    }


    with open(
        HPO_DIR
        / f"H{H:02d}_SELECTED_CONFIGURATION.json",
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            selection,
            f,
            indent=4
        )


    summary_rows.append(
        {
            "Horizon":
                H,

            "Trial":
                selected.number,

            "Mean_CV_AP":
                selected.value,

            "CV_AP_Std":
                selected.user_attrs[
                    "std_ap"
                ],

            "Mean_CV_F1":
                selected.user_attrs[
                    "mean_f1"
                ],

            "Mean_CV_MCC":
                selected.user_attrs[
                    "mean_mcc"
                ],

            "Mean_Best_Iteration":
                selected.user_attrs[
                    "mean_best_iteration"
                ],
        }
    )


    print("\nSELECTED:")

    print(
        json.dumps(
            selection,
            indent=4
        )
    )


HPO_SUMMARY = pd.DataFrame(
    summary_rows
)


HPO_SUMMARY.to_csv(
    HPO_DIR
    / "CLEAN_CV_HPO_SUMMARY.csv",
    index=False
)


print("\n" + "=" * 120)
print("✓ CELL 3 COMPLETE")
print("=" * 120)

print(
    HPO_SUMMARY.to_string(
        index=False
    )
)

print(
    "\nOfficial VALIDATION loaded: NO ✓"
)

print(
    "TEST loaded: NO ✓"
)

print(
    "\nRun CELL 4 next."
)

In [ ]:
# ==================================================================================================
# XGBOOST NOTEBOOK — CELL 4
#
# SELECTED XGBOOST CONFIG
# FULL TRAIN → OFFICIAL VALIDATION
#
# THREE FROZEN SEEDS
#
# TEST IS NOT DEFINED OR LOADED.
# ==================================================================================================

from pathlib import Path
import json
import shutil
import gc

import numpy as np
import pandas as pd

import xgboost as xgb

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    accuracy_score,
    balanced_accuracy_score,
    matthews_corrcoef,
    brier_score_loss,
    confusion_matrix,
    precision_recall_curve,
    auc,
)


print("=" * 120)
print("CELL 4 — FINAL XGBOOST TRAIN → OFFICIAL VALIDATION")
print("=" * 120)


# ==================================================================================================
# PATHS
# ==================================================================================================

ROOT = Path(
    "/content/drive/MyDrive/network-attack-forecasting/Synthetic_NAF_v1"
)

P12_0 = (
    ROOT
    / "model_ready"
    / "12_0"
)

MODEL_ROOT = (
    ROOT
    / "models"
    / "XGBOOST_W10_TEMPORAL_SUMMARY"
)

FEATURE_DIR = (
    MODEL_ROOT
    / "temporal_features"
)

HPO_DIR = (
    MODEL_ROOT
    / "hpo"
)

FINAL_DIR = (
    MODEL_ROOT
    / "final_validation"
)

FINAL_DIR.mkdir(
    parents=True,
    exist_ok=True
)


TRAIN_X_FILE = (
    FEATURE_DIR
    / "X_train_log1p_temporal98.npy"
)

VAL_X_FILE = (
    FEATURE_DIR
    / "X_validation_log1p_temporal98.npy"
)

FEATURE_NAMES_FILE = (
    FEATURE_DIR
    / "feature_names.json"
)


TRAIN_Y_FILE = (
    P12_0
    / "W10"
    / "train"
    / "y_binary.npy"
)

TRAIN_ELIGIBLE_FILE = (
    P12_0
    / "W10"
    / "train"
    / "eligible.npy"
)


VAL_Y_FILE = (
    P12_0
    / "W10"
    / "validation"
    / "y_binary.npy"
)

VAL_ELIGIBLE_FILE = (
    P12_0
    / "W10"
    / "validation"
    / "eligible.npy"
)

VAL_INDEX_FILE = (
    P12_0
    / "W10"
    / "validation"
    / "index.parquet"
)


# ==================================================================================================
# CONSTANTS
# ==================================================================================================

HORIZONS = [
    5,
    10,
]

HORIZON_TO_COLUMN = {
    1: 0,
    5: 1,
    10: 2,
    15: 3,
    30: 4,
}

FINAL_SEEDS = [
    20260930,
    20260931,
    20260932,
]


# ==================================================================================================
# GPU / DEVICE
# ==================================================================================================

GPU_AVAILABLE = (
    shutil.which(
        "nvidia-smi"
    )
    is not None
)


XGB_VERSION_MAJOR = int(
    xgb.__version__.split(
        "."
    )[0]
)


def device_params():

    if GPU_AVAILABLE:

        if XGB_VERSION_MAJOR >= 2:

            return {
                "tree_method":
                    "hist",

                "device":
                    "cuda",
            }

        return {
            "tree_method":
                "gpu_hist",
        }

    return {
        "tree_method":
            "hist",

        "nthread":
            -1,
    }


print(
    "GPU available:",
    GPU_AVAILABLE
)

print(
    "Device:",
    device_params()
)


# ==================================================================================================
# LOAD
# ==================================================================================================

X_train_all = np.load(
    TRAIN_X_FILE,
    mmap_mode="r"
)

X_val_all = np.load(
    VAL_X_FILE,
    mmap_mode="r"
)


Y_train_all = np.load(
    TRAIN_Y_FILE,
    allow_pickle=False
)

eligible_train = np.load(
    TRAIN_ELIGIBLE_FILE,
    allow_pickle=False
)


Y_val_all = np.load(
    VAL_Y_FILE,
    allow_pickle=False
)

eligible_val = np.load(
    VAL_ELIGIBLE_FILE,
    allow_pickle=False
)


val_index_all = pd.read_parquet(
    VAL_INDEX_FILE
)


with open(
    FEATURE_NAMES_FILE,
    "r",
    encoding="utf-8"
) as f:

    FEATURE_NAMES = json.load(
        f
    )


val_index_all = val_index_all.copy()

val_index_all[
    "Episode_ID"
] = (
    val_index_all[
        "Episode_ID"
    ]
    .astype(str)
)


# ==================================================================================================
# HELPERS
# ==================================================================================================

def calculate_scale_pos_weight(
    y,
    strategy
):

    pos = float(
        (
            y
            ==
            1
        ).sum()
    )

    neg = float(
        (
            y
            ==
            0
        ).sum()
    )


    ratio = (
        neg
        /
        pos
    )


    if strategy == "NONE":

        return 1.0


    if strategy == "SQRT":

        return float(
            np.sqrt(
                ratio
            )
        )


    if strategy == "FULL":

        return float(
            ratio
        )


    raise ValueError(
        strategy
    )


def choose_f1_threshold(
    y,
    probability
):

    precision, recall, thresholds = (
        precision_recall_curve(
            y,
            probability
        )
    )


    if len(
        thresholds
    ) == 0:

        return 0.5


    precision = precision[
        :-1
    ]

    recall = recall[
        :-1
    ]


    denom = (
        precision
        +
        recall
    )


    f1 = np.divide(
        2
        *
        precision
        *
        recall,
        denom,
        out=np.zeros_like(
            denom
        ),
        where=(
            denom
            >
            0
        )
    )


    best = f1.max()


    candidates = np.flatnonzero(
        np.isclose(
            f1,
            best,
            atol=1e-12,
            rtol=0
        )
    )


    table = pd.DataFrame(
        {
            "i":
                candidates,

            "precision":
                precision[
                    candidates
                ],

            "threshold":
                thresholds[
                    candidates
                ]
        }
    )


    table = table.sort_values(
        [
            "precision",
            "threshold"
        ],
        ascending=[
            False,
            False
        ]
    )


    i = int(
        table.iloc[
            0
        ][
            "i"
        ]
    )


    return float(
        thresholds[
            i
        ]
    )


def calculate_metrics(
    y,
    probability,
    threshold
):

    prediction = (
        probability
        >=
        threshold
    ).astype(
        np.int8
    )


    tn, fp, fn, tp = confusion_matrix(
        y,
        prediction,
        labels=[
            0,
            1
        ]
    ).ravel()


    specificity = (
        tn
        /
        (
            tn
            +
            fp
        )
    )


    precision_curve, recall_curve, _ = (
        precision_recall_curve(
            y,
            probability
        )
    )


    return {
        "Average_Precision":
            float(
                average_precision_score(
                    y,
                    probability
                )
            ),

        "PR_AUC":
            float(
                auc(
                    recall_curve,
                    precision_curve
                )
            ),

        "ROC_AUC":
            float(
                roc_auc_score(
                    y,
                    probability
                )
            ),

        "Accuracy":
            float(
                accuracy_score(
                    y,
                    prediction
                )
            ),

        "Balanced_Accuracy":
            float(
                balanced_accuracy_score(
                    y,
                    prediction
                )
            ),

        "Precision":
            float(
                precision_score(
                    y,
                    prediction,
                    zero_division=0
                )
            ),

        "Recall":
            float(
                recall_score(
                    y,
                    prediction,
                    zero_division=0
                )
            ),

        "Specificity":
            float(
                specificity
            ),

        "F1":
            float(
                f1_score(
                    y,
                    prediction,
                    zero_division=0
                )
            ),

        "MCC":
            float(
                matthews_corrcoef(
                    y,
                    prediction
                )
            ),

        "Brier":
            float(
                brier_score_loss(
                    y,
                    probability
                )
            ),

        "Threshold":
            float(
                threshold
            ),

        "TN":
            int(
                tn
            ),

        "FP":
            int(
                fp
            ),

        "FN":
            int(
                fn
            ),

        "TP":
            int(
                tp
            ),
    }


def calculate_event_metrics(
    y,
    probability,
    threshold,
    metadata
):

    prediction = (
        probability
        >=
        threshold
    )


    episodes = (
        metadata[
            "Episode_ID"
        ]
        .astype(str)
        .to_numpy()
    )


    tto = pd.to_numeric(
        metadata[
            "Minutes_To_Next_Onset"
        ],
        errors="coerce"
    ).to_numpy(
        dtype=float
    )


    positive = (
        y
        ==
        1
    )


    negative = (
        y
        ==
        0
    )


    true_positive_warning = (
        prediction
        &
        positive
    )


    false_positive_warning = (
        prediction
        &
        negative
    )


    true_events = set(
        episodes[
            positive
        ]
    )


    recalled_events = set(
        episodes[
            true_positive_warning
        ]
    )


    warned_episodes = set(
        episodes[
            prediction
        ]
    )


    event_recall = (
        len(
            recalled_events
        )
        /
        len(
            true_events
        )
    )


    event_precision = (
        len(
            recalled_events
        )
        /
        len(
            warned_episodes
        )

        if warned_episodes

        else np.nan
    )


    leads = []


    for episode_id in recalled_events:

        mask = (

            true_positive_warning

            &

            (
                episodes
                ==
                episode_id
            )
        )


        values = tto[
            mask
        ]


        values = values[
            np.isfinite(
                values
            )
        ]


        if len(
            values
        ):

            leads.append(
                float(
                    np.max(
                        values
                    )
                )
            )


    negative_hours = (
        negative.sum()
        /
        60.0
    )


    fp_per_hour = (
        false_positive_warning.sum()
        /
        negative_hours
    )


    return {
        "Event_Recall":
            float(
                event_recall
            ),

        "Event_Precision":
            float(
                event_precision
            ),

        "Median_First_Warning_Lead_Min":
            (
                float(
                    np.median(
                        leads
                    )
                )

                if leads

                else np.nan
            ),

        "Mean_First_Warning_Lead_Min":
            (
                float(
                    np.mean(
                        leads
                    )
                )

                if leads

                else np.nan
            ),

        "False_Warning_Origins_Per_Hour":
            float(
                fp_per_hour
            ),
    }


# ==================================================================================================
# TRAIN
# ==================================================================================================

all_rows = []


for H in HORIZONS:

    print("\n" + "#" * 120)
    print(
        f"FINAL XGBOOST DEVELOPMENT — H{H}"
    )
    print("#" * 120)


    selection_file = (
        HPO_DIR
        / f"H{H:02d}_SELECTED_CONFIGURATION.json"
    )


    if not selection_file.is_file():

        raise FileNotFoundError(
            f"Run Cell 3 first:\n{selection_file}"
        )


    with open(
        selection_file,
        "r",
        encoding="utf-8"
    ) as f:

        selection = json.load(
            f
        )


    config = selection[
        "Configuration"
    ]


    print(
        "Selected CV configuration:"
    )

    print(
        json.dumps(
            config,
            indent=4
        )
    )


    h_col = HORIZON_TO_COLUMN[
        H
    ]


    train_mask = (
        eligible_train[
            :,
            h_col
        ]
        ==
        1
    )


    val_mask = (
        eligible_val[
            :,
            h_col
        ]
        ==
        1
    )


    X_train = np.asarray(
        X_train_all[
            train_mask
        ],
        dtype=np.float32
    )


    y_train = np.asarray(
        Y_train_all[
            train_mask,
            h_col
        ],
        dtype=np.int8
    )


    X_val = np.asarray(
        X_val_all[
            val_mask
        ],
        dtype=np.float32
    )


    y_val = np.asarray(
        Y_val_all[
            val_mask,
            h_col
        ],
        dtype=np.int8
    )


    val_metadata = (
        val_index_all
        .loc[
            val_mask
        ]
        .reset_index(
            drop=True
        )
    )


    print(
        "\nTRAIN:",
        X_train.shape,
        "positive=",
        int(
            y_train.sum()
        ),
        "negative=",
        int(
            (
                y_train
                ==
                0
            ).sum()
        )
    )


    print(
        "VALIDATION:",
        X_val.shape,
        "positive=",
        int(
            y_val.sum()
        ),
        "negative=",
        int(
            (
                y_val
                ==
                0
            ).sum()
        )
    )


    horizon_dir = (
        FINAL_DIR
        / f"H{H:02d}"
    )

    horizon_dir.mkdir(
        parents=True,
        exist_ok=True
    )


    for seed in FINAL_SEEDS:

        print("\n" + "-" * 120)

        print(
            f"H{H} | seed={seed}"
        )

        print("-" * 120)


        scale_pos_weight = calculate_scale_pos_weight(
            y_train,
            config[
                "scale_pos_weight_strategy"
            ]
        )


        dtrain = xgb.DMatrix(
            X_train,
            label=y_train,
            feature_names=FEATURE_NAMES
        )


        dval = xgb.DMatrix(
            X_val,
            label=y_val,
            feature_names=FEATURE_NAMES
        )


        params = {
            "objective":
                "binary:logistic",

            "eval_metric":
                "aucpr",

            "max_depth":
                config[
                    "max_depth"
                ],

            "min_child_weight":
                config[
                    "min_child_weight"
                ],

            "eta":
                config[
                    "learning_rate"
                ],

            "subsample":
                config[
                    "subsample"
                ],

            "colsample_bytree":
                config[
                    "colsample_bytree"
                ],

            "gamma":
                config[
                    "gamma"
                ],

            "alpha":
                config[
                    "reg_alpha"
                ],

            "lambda":
                config[
                    "reg_lambda"
                ],

            "scale_pos_weight":
                scale_pos_weight,

            "max_bin":
                256,

            "seed":
                seed,

            "verbosity":
                0,

            **device_params(),
        }


        evaluation_results = {}


        model = xgb.train(
            params=params,
            dtrain=dtrain,
            num_boost_round=3000,
            evals=[
                (
                    dval,
                    "validation"
                )
            ],
            early_stopping_rounds=150,
            evals_result=evaluation_results,
            verbose_eval=False,
        )


        best_iteration = int(
            model.best_iteration
        )


        probability = model.predict(
            dval,
            iteration_range=(
                0,
                best_iteration
                +
                1
            )
        )


        threshold = choose_f1_threshold(
            y_val,
            probability
        )


        metrics = calculate_metrics(
            y_val,
            probability,
            threshold
        )


        event_metrics = calculate_event_metrics(
            y_val,
            probability,
            threshold,
            val_metadata
        )


        row = {
            "Model":
                "XGBOOST_W10_TEMPORAL_SUMMARY",

            "Horizon":
                H,

            "Seed":
                seed,

            "Best_Iteration":
                best_iteration,

            "Scale_Pos_Weight":
                scale_pos_weight,

            **config,

            **metrics,

            **event_metrics,

            "TEST_Used":
                False,
        }


        all_rows.append(
            row
        )


        seed_dir = (
            horizon_dir
            / f"seed_{seed}"
        )

        seed_dir.mkdir(
            parents=True,
            exist_ok=True
        )


        model.save_model(
            seed_dir
            / "model.json"
        )


        with open(
            seed_dir
            / "validation_metrics.json",
            "w",
            encoding="utf-8"
        ) as f:

            json.dump(
                row,
                f,
                indent=4
            )


        prediction_df = val_metadata[
            [
                "Episode_ID",
                "Minute_Index",
                "Minutes_To_Next_Onset",
            ]
        ].copy()


        prediction_df[
            "Y_True"
        ] = y_val


        prediction_df[
            "Probability"
        ] = probability


        prediction_df[
            "Threshold"
        ] = threshold


        prediction_df[
            "Prediction"
        ] = (
            probability
            >=
            threshold
        ).astype(
            np.int8
        )


        prediction_df.to_parquet(
            seed_dir
            / "validation_predictions.parquet",
            index=False
        )


        pd.DataFrame(
            evaluation_results[
                "validation"
            ]
        ).to_csv(
            seed_dir
            / "training_curve.csv",
            index=False
        )


        # ------------------------------------------------------------------------------------------
        # Feature importance
        # ------------------------------------------------------------------------------------------

        gain = model.get_score(
            importance_type="gain"
        )


        importance = pd.DataFrame(
            {
                "Feature":
                    FEATURE_NAMES,

                "Gain":
                    [
                        gain.get(
                            feature,
                            0.0
                        )
                        for feature
                        in FEATURE_NAMES
                    ]
            }
        ).sort_values(
            "Gain",
            ascending=False
        )


        importance.to_csv(
            seed_dir
            / "feature_importance_gain.csv",
            index=False
        )


        print(
            f"Best iteration = {best_iteration}"
        )

        print(
            f"AP             = {metrics['Average_Precision']:.6f}"
        )

        print(
            f"PR-AUC         = {metrics['PR_AUC']:.6f}"
        )

        print(
            f"ROC-AUC        = {metrics['ROC_AUC']:.6f}"
        )

        print(
            f"Accuracy       = {metrics['Accuracy']:.6f}"
        )

        print(
            f"Balanced Acc   = {metrics['Balanced_Accuracy']:.6f}"
        )

        print(
            f"Precision      = {metrics['Precision']:.6f}"
        )

        print(
            f"Recall         = {metrics['Recall']:.6f}"
        )

        print(
            f"F1             = {metrics['F1']:.6f}"
        )

        print(
            f"MCC            = {metrics['MCC']:.6f}"
        )

        print(
            f"Event Recall   = {event_metrics['Event_Recall']:.6f}"
        )

        print(
            f"Event Precision= {event_metrics['Event_Precision']:.6f}"
        )

        print(
            f"Median Lead    = {event_metrics['Median_First_Warning_Lead_Min']}"
        )

        print(
            f"FP origins/hr  = {event_metrics['False_Warning_Origins_Per_Hour']:.6f}"
        )


# ==================================================================================================
# THREE-SEED SUMMARY
# ==================================================================================================

results = pd.DataFrame(
    all_rows
)


results.to_csv(
    FINAL_DIR
    / "ALL_VALIDATION_REPLICATES.csv",
    index=False
)


summary_rows = []


for H in HORIZONS:

    df = results.loc[
        results[
            "Horizon"
        ]
        ==
        H
    ]


    summary_rows.append(
        {
            "Horizon":
                H,

            "Average_Precision_Mean":
                float(
                    df[
                        "Average_Precision"
                    ].mean()
                ),

            "Average_Precision_Std":
                float(
                    df[
                        "Average_Precision"
                    ].std(
                        ddof=1
                    )
                ),

            "PR_AUC_Mean":
                float(
                    df[
                        "PR_AUC"
                    ].mean()
                ),

            "ROC_AUC_Mean":
                float(
                    df[
                        "ROC_AUC"
                    ].mean()
                ),

            "Accuracy_Mean":
                float(
                    df[
                        "Accuracy"
                    ].mean()
                ),

            "Balanced_Accuracy_Mean":
                float(
                    df[
                        "Balanced_Accuracy"
                    ].mean()
                ),

            "Precision_Mean":
                float(
                    df[
                        "Precision"
                    ].mean()
                ),

            "Recall_Mean":
                float(
                    df[
                        "Recall"
                    ].mean()
                ),

            "F1_Mean":
                float(
                    df[
                        "F1"
                    ].mean()
                ),

            "MCC_Mean":
                float(
                    df[
                        "MCC"
                    ].mean()
                ),

            "Brier_Mean":
                float(
                    df[
                        "Brier"
                    ].mean()
                ),

            "Event_Recall_Mean":
                float(
                    df[
                        "Event_Recall"
                    ].mean()
                ),

            "Event_Precision_Mean":
                float(
                    df[
                        "Event_Precision"
                    ].mean()
                ),

            "Median_First_Warning_Lead_Min_Mean":
                float(
                    df[
                        "Median_First_Warning_Lead_Min"
                    ].mean()
                ),

            "False_Warning_Origins_Per_Hour_Mean":
                float(
                    df[
                        "False_Warning_Origins_Per_Hour"
                    ].mean()
                ),
        }
    )


summary = pd.DataFrame(
    summary_rows
)


summary.to_csv(
    FINAL_DIR
    / "XGBOOST_VALIDATION_SUMMARY.csv",
    index=False
)


print("\n" + "=" * 120)
print("XGBOOST — OFFICIAL VALIDATION SUMMARY")
print("=" * 120)


print(
    summary.to_string(
        index=False
    )
)


# ==================================================================================================
# DIRECT LSTM COMPARISON
# ==================================================================================================

LSTM_SUMMARY_FILE = (
    ROOT
    / "models"
    / "LSTM_W10_PRIMARY"
    / "validation_diagnostics"
    / "LSTM_FULL_VALIDATION_SUMMARY.csv"
)


if LSTM_SUMMARY_FILE.is_file():

    lstm = pd.read_csv(
        LSTM_SUMMARY_FILE
    )


    compare_rows = []


    for H in HORIZONS:

        xgb_row = summary.loc[
            summary[
                "Horizon"
            ]
            ==
            H
        ].iloc[
            0
        ]


        lstm_row = lstm.loc[
            lstm[
                "Horizon"
            ]
            ==
            H
        ].iloc[
            0
        ]


        compare_rows.append(
            {
                "Horizon":
                    H,

                "LSTM_AP":
                    float(
                        lstm_row[
                            "Average_Precision_Mean"
                        ]
                    ),

                "XGBoost_AP":
                    float(
                        xgb_row[
                            "Average_Precision_Mean"
                        ]
                    ),

                "AP_Change_XGB_minus_LSTM":
                    float(
                        xgb_row[
                            "Average_Precision_Mean"
                        ]
                        -
                        lstm_row[
                            "Average_Precision_Mean"
                        ]
                    ),

                "LSTM_F1":
                    float(
                        lstm_row[
                            "F1_Mean"
                        ]
                    ),

                "XGBoost_F1":
                    float(
                        xgb_row[
                            "F1_Mean"
                        ]
                    ),

                "F1_Change":
                    float(
                        xgb_row[
                            "F1_Mean"
                        ]
                        -
                        lstm_row[
                            "F1_Mean"
                        ]
                    ),

                "LSTM_Event_Recall":
                    float(
                        lstm_row[
                            "Event_Recall_Mean"
                        ]
                    ),

                "XGBoost_Event_Recall":
                    float(
                        xgb_row[
                            "Event_Recall_Mean"
                        ]
                    ),
            }
        )


    comparison = pd.DataFrame(
        compare_rows
    )


    comparison.to_csv(
        FINAL_DIR
        / "XGBOOST_VS_LSTM.csv",
        index=False
    )


    print("\n" + "=" * 120)
    print("XGBOOST VS ORIGINAL LSTM")
    print("=" * 120)


    print(
        comparison.to_string(
            index=False
        )
    )


# ==================================================================================================
# DEVELOPMENT LOCK
# ==================================================================================================

with open(
    MODEL_ROOT
    / "XGBOOST_DEVELOPMENT_LOCK.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        {
            "Model":
                "XGBOOST_W10_TEMPORAL_SUMMARY",

            "Status":
                "TRAIN_VALIDATION_DEVELOPMENT_COMPLETE",

            "HPO":
                "3-fold TRAIN episode CV",

            "Official_Validation_Used_For_HPO":
                False,

            "Official_Validation_Used_For_Final_Development":
                True,

            "Final_Seeds":
                FINAL_SEEDS,

            "TEST_Path_Defined":
                False,

            "TEST_Loaded":
                False,

            "TEST_Used":
                False,

            "TEST_Evaluation_Performed":
                False,
        },
        f,
        indent=4
    )


print("\n" + "=" * 120)
print("✓ CELL 4 COMPLETE")
print("=" * 120)

print(
    "Three-seed XGBoost development complete ✓"
)

print(
    "Direct LSTM comparison saved ✓"
)

print(
    "TEST path defined: NO ✓"
)

print(
    "TEST loaded: NO ✓"
)

print(
    "TEST evaluated: NO ✓"
)

print(
    "\nDO NOT RUN TEST YET."
)

print("=" * 120)

gc.collect()

In [ ]:
# ==================================================================================================
# XGBOOST — CELL 5A
#
# V2 REPRESENTATION OPTIMIZATION INFRASTRUCTURE
#
# BUILDS:
#
#   1. SUMMARY98
#   2. FLAT140
#   3. HYBRID238
#   4. HYBRID_DIFF364
#
# CV SPLIT:
#   Episode-level
#   Attempts to balance:
#       Episode_Type
#       Attack_Family
#       Calibration_Source
#       Precursor_Regime
#
# OFFICIAL VALIDATION IS NOT USED FOR HPO.
# TEST PATH IS NEVER DEFINED.
# ==================================================================================================

from pathlib import Path
import json
import hashlib
import shutil
import gc

import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold


print("=" * 120)
print("XGBOOST V2 — CELL 5A")
print("ORDERED + HYBRID TEMPORAL REPRESENTATIONS")
print("=" * 120)


# ==================================================================================================
# PATHS
# ==================================================================================================

ROOT = Path(
    "/content/drive/MyDrive/network-attack-forecasting/Synthetic_NAF_v1"
)

P12_0 = (
    ROOT
    / "model_ready"
    / "12_0"
)

P12_3 = (
    ROOT
    / "model_ready"
    / "12_3"
)

V1_ROOT = (
    ROOT
    / "models"
    / "XGBOOST_W10_TEMPORAL_SUMMARY"
)

V2_ROOT = (
    ROOT
    / "models"
    / "XGBOOST_W10_TEMPORAL_V2"
)

REP_DIR = (
    V2_ROOT
    / "representations"
)

HPO_DIR = (
    V2_ROOT
    / "hpo"
)

REP_DIR.mkdir(
    parents=True,
    exist_ok=True
)

HPO_DIR.mkdir(
    parents=True,
    exist_ok=True
)


TRAIN_X_RAW_FILE = (
    P12_0
    / "W10"
    / "train"
    / "X.npy"
)

TRAIN_Y_FILE = (
    P12_0
    / "W10"
    / "train"
    / "y_binary.npy"
)

TRAIN_ELIGIBLE_FILE = (
    P12_0
    / "W10"
    / "train"
    / "eligible.npy"
)

TRAIN_INDEX_FILE = (
    P12_0
    / "W10"
    / "train"
    / "index.parquet"
)


READY_FILE = (
    P12_3
    / "12_3_READY_FOR_MODELING.json"
)

LOCK_FILE = (
    P12_3
    / "12_3_PROTOCOL_LOCK.json"
)

LOCK_HASH_FILE = (
    P12_3
    / "12_3_PROTOCOL_LOCK_SHA256.txt"
)


# ==================================================================================================
# CONSTANTS
# ==================================================================================================

WINDOW = 10

FEATURE_COUNT = 14

CV_FOLDS = 3

CV_SEED = 20261022


BASE_FEATURES = [
    "Flow_Count",
    "Flow_Duration_Mean",
    "Flow_Duration_Std",
    "Flow_Duration_Max",
    "Forward_Packets_Sum",
    "Backward_Packets_Sum",
    "Total_Packets",
    "Forward_Bytes_Sum",
    "Backward_Bytes_Sum",
    "Total_Bytes",
    "Packets_Per_Second",
    "Bytes_Per_Second",
    "Forward_Backward_Packet_Ratio",
    "Forward_Backward_Byte_Ratio",
]


REPRESENTATIONS = [
    "SUMMARY98",
    "FLAT140",
    "HYBRID238",
    "HYBRID_DIFF364",
]


# ==================================================================================================
# HELPERS
# ==================================================================================================

def require_file(path):

    if not path.is_file():

        raise FileNotFoundError(
            f"Missing required file:\n{path}"
        )


def sha256_file(path):

    h = hashlib.sha256()

    with open(
        path,
        "rb"
    ) as f:

        while True:

            chunk = f.read(
                1024 * 1024
            )

            if not chunk:

                break

            h.update(
                chunk
            )

    return h.hexdigest()


def read_json(path):

    require_file(
        path
    )

    with open(
        path,
        "r",
        encoding="utf-8"
    ) as f:

        return json.load(
            f
        )


def write_json(
    obj,
    path
):

    path.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    with open(
        path,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            obj,
            f,
            indent=4
        )


# ==================================================================================================
# VERIFY FROZEN PROTOCOL
# ==================================================================================================

for path in [
    READY_FILE,
    LOCK_FILE,
    LOCK_HASH_FILE,
    TRAIN_X_RAW_FILE,
    TRAIN_Y_FILE,
    TRAIN_ELIGIBLE_FILE,
    TRAIN_INDEX_FILE,
]:

    require_file(
        path
    )


ready = read_json(
    READY_FILE
)


if ready.get(
    "phase_12_3_status"
) != "PASS":

    raise RuntimeError(
        "Phase 12.3 is not PASS."
    )


if ready.get(
    "training_authorized"
) is not True:

    raise RuntimeError(
        "Training not authorized."
    )


if ready.get(
    "test_evaluation_authorized"
) is not False:

    raise RuntimeError(
        "Unexpected TEST authorization."
    )


protocol_hash = sha256_file(
    LOCK_FILE
)


expected_protocol_hash = (
    LOCK_HASH_FILE
    .read_text(
        encoding="utf-8"
    )
    .strip()
)


if protocol_hash != expected_protocol_hash:

    raise RuntimeError(
        "Protocol hash mismatch."
    )


print(
    "✓ Phase 12.3 verified"
)

print(
    "✓ TRAIN development authorized"
)

print(
    "✓ TEST embargo intact"
)


# ==================================================================================================
# LOAD RAW TRAIN
# ==================================================================================================

X_RAW = np.load(
    TRAIN_X_RAW_FILE,
    mmap_mode="r"
)


if X_RAW.shape != (
    59837,
    10,
    14
):

    raise RuntimeError(
        f"Unexpected TRAIN shape: {X_RAW.shape}"
    )


if not np.isfinite(
    X_RAW
).all():

    raise RuntimeError(
        "Non-finite raw predictor."
    )


if np.any(
    X_RAW
    <
    0
):

    raise RuntimeError(
        "Negative raw value prevents log1p."
    )


print(
    "\nRaw TRAIN:",
    X_RAW.shape
)


# ==================================================================================================
# LOG1P W10
# ==================================================================================================

X_LOG = np.log1p(
    np.asarray(
        X_RAW,
        dtype=np.float32
    )
).astype(
    np.float32
)


# ==================================================================================================
# SUMMARY98
# ==================================================================================================

last = X_LOG[
    :,
    -1,
    :
]


mean = X_LOG.mean(
    axis=1
)


std = X_LOG.std(
    axis=1,
    ddof=0
)


minimum = X_LOG.min(
    axis=1
)


maximum = X_LOG.max(
    axis=1
)


delta = (
    X_LOG[
        :,
        -1,
        :
    ]
    -
    X_LOG[
        :,
        0,
        :
    ]
)


t = np.arange(
    WINDOW,
    dtype=np.float32
)


t_centered = (
    t
    -
    t.mean()
)


slope_denominator = np.sum(
    t_centered ** 2
)


slope = (

    np.sum(
        X_LOG
        *
        t_centered[
            None,
            :,
            None
        ],
        axis=1
    )

    /

    slope_denominator
)


SUMMARY98 = np.concatenate(
    [
        last,
        mean,
        std,
        minimum,
        maximum,
        delta,
        slope,
    ],
    axis=1
).astype(
    np.float32
)


# ==================================================================================================
# FLAT140
#
# Exact temporal ordering:
#
# t-9 features 1..14
# t-8 features 1..14
# ...
# t   features 1..14
# ==================================================================================================

FLAT140 = X_LOG.reshape(
    len(
        X_LOG
    ),
    -1
).astype(
    np.float32
)


# ==================================================================================================
# EXACT FIRST DIFFERENCES
#
# 9 transitions × 14 features = 126
# ==================================================================================================

DIFF126 = np.diff(
    X_LOG,
    axis=1
).reshape(
    len(
        X_LOG
    ),
    -1
).astype(
    np.float32
)


# ==================================================================================================
# HYBRIDS
# ==================================================================================================

HYBRID238 = np.concatenate(
    [
        FLAT140,
        SUMMARY98,
    ],
    axis=1
).astype(
    np.float32
)


HYBRID_DIFF364 = np.concatenate(
    [
        FLAT140,
        SUMMARY98,
        DIFF126,
    ],
    axis=1
).astype(
    np.float32
)


# ==================================================================================================
# SHAPE ASSERTIONS
# ==================================================================================================

expected_shapes = {
    "SUMMARY98":
        98,

    "FLAT140":
        140,

    "HYBRID238":
        238,

    "HYBRID_DIFF364":
        364,
}


representation_arrays = {
    "SUMMARY98":
        SUMMARY98,

    "FLAT140":
        FLAT140,

    "HYBRID238":
        HYBRID238,

    "HYBRID_DIFF364":
        HYBRID_DIFF364,
}


for name, array in representation_arrays.items():

    if array.shape != (
        59837,
        expected_shapes[
            name
        ]
    ):

        raise RuntimeError(
            f"{name} shape wrong: {array.shape}"
        )


    if not np.isfinite(
        array
    ).all():

        raise RuntimeError(
            f"{name} contains non-finite values."
        )


# ==================================================================================================
# FEATURE NAMES
# ==================================================================================================

SUMMARY_NAMES = []


for statistic in [
    "LAST",
    "MEAN",
    "STD",
    "MIN",
    "MAX",
    "DELTA",
    "SLOPE",
]:

    for feature in BASE_FEATURES:

        SUMMARY_NAMES.append(
            f"{statistic}__{feature}"
        )


FLAT_NAMES = []


for position in range(
    WINDOW
):

    minutes_back = (
        WINDOW
        -
        1
        -
        position
    )


    prefix = (
        "T"
        if minutes_back == 0
        else f"T_MINUS_{minutes_back}"
    )


    for feature in BASE_FEATURES:

        FLAT_NAMES.append(
            f"{prefix}__{feature}"
        )


DIFF_NAMES = []


for position in range(
    WINDOW
    -
    1
):

    from_back = (
        WINDOW
        -
        1
        -
        position
    )


    to_back = (
        from_back
        -
        1
    )


    from_name = (
        "T"
        if from_back == 0
        else f"T_MINUS_{from_back}"
    )


    to_name = (
        "T"
        if to_back == 0
        else f"T_MINUS_{to_back}"
    )


    for feature in BASE_FEATURES:

        DIFF_NAMES.append(
            f"DIFF_{from_name}_TO_{to_name}__{feature}"
        )


REPRESENTATION_NAMES = {
    "SUMMARY98":
        SUMMARY_NAMES,

    "FLAT140":
        FLAT_NAMES,

    "HYBRID238":
        (
            FLAT_NAMES
            +
            SUMMARY_NAMES
        ),

    "HYBRID_DIFF364":
        (
            FLAT_NAMES
            +
            SUMMARY_NAMES
            +
            DIFF_NAMES
        ),
}


for representation in REPRESENTATIONS:

    if len(
        REPRESENTATION_NAMES[
            representation
        ]
    ) != expected_shapes[
        representation
    ]:

        raise RuntimeError(
            f"{representation}: feature-name mismatch."
        )


# ==================================================================================================
# SAVE REPRESENTATIONS
# ==================================================================================================

print("\n" + "=" * 120)
print("SAVING V2 REPRESENTATIONS")
print("=" * 120)


for representation in REPRESENTATIONS:

    array_path = (
        REP_DIR
        / f"TRAIN_{representation}.npy"
    )


    names_path = (
        REP_DIR
        / f"{representation}_FEATURE_NAMES.json"
    )


    np.save(
        array_path,
        representation_arrays[
            representation
        ]
    )


    with open(
        names_path,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            REPRESENTATION_NAMES[
                representation
            ],
            f,
            indent=4
        )


    print(
        f"✓ {representation}: "
        f"{representation_arrays[representation].shape}"
    )


# ==================================================================================================
# EPISODE TABLE
# ==================================================================================================

INDEX = pd.read_parquet(
    TRAIN_INDEX_FILE
)


INDEX = INDEX.copy()


INDEX[
    "Episode_ID"
] = (
    INDEX[
        "Episode_ID"
    ]
    .astype(str)
)


for column in [
    "Episode_Type",
    "Episode_Attack_Family",
    "Calibration_Source",
    "Precursor_Regime",
]:

    if column not in INDEX.columns:

        INDEX[
            column
        ] = "NONE"


    INDEX[
        column
    ] = (
        INDEX[
            column
        ]
        .astype("string")
        .fillna("NONE")
        .astype(str)
        .str.strip()
    )


episode_table = (

    INDEX[
        [
            "Episode_ID",
            "Episode_Type",
            "Episode_Attack_Family",
            "Calibration_Source",
            "Precursor_Regime",
        ]
    ]

    .groupby(
        "Episode_ID",
        as_index=False
    )

    .first()
)


if len(
    episode_table
) != 840:

    raise RuntimeError(
        f"Expected 840 episodes, got {len(episode_table)}"
    )


# ==================================================================================================
# STRATIFICATION
#
# Try richest combination first.
# Fall back only when a stratum has <3 episodes.
# ==================================================================================================

stratification_candidates = [

    [
        "Episode_Type",
        "Episode_Attack_Family",
        "Calibration_Source",
        "Precursor_Regime",
    ],

    [
        "Episode_Type",
        "Episode_Attack_Family",
        "Precursor_Regime",
    ],

    [
        "Episode_Type",
        "Episode_Attack_Family",
        "Calibration_Source",
    ],

    [
        "Episode_Type",
        "Episode_Attack_Family",
    ],
]


STRAT_COLUMNS = None


for candidate_columns in stratification_candidates:

    candidate = (
        episode_table[
            candidate_columns
        ]
        .astype(str)
        .agg(
            "|".join,
            axis=1
        )
    )


    counts = candidate.value_counts()


    if (
        counts
        >=
        CV_FOLDS
    ).all():

        STRAT_COLUMNS = candidate_columns

        episode_table[
            "CV_Stratum"
        ] = candidate

        break


if STRAT_COLUMNS is None:

    raise RuntimeError(
        "Unable to construct valid 3-fold episode strata."
    )


print("\n" + "=" * 120)
print("EPISODE CV STRATIFICATION")
print("=" * 120)


print(
    "Selected stratification columns:"
)

print(
    STRAT_COLUMNS
)


print(
    "\nNumber of strata:",
    episode_table[
        "CV_Stratum"
    ].nunique()
)


print(
    "Smallest stratum:",
    episode_table[
        "CV_Stratum"
    ].value_counts().min()
)


# ==================================================================================================
# CREATE FIXED CV SPLITS
# ==================================================================================================

skf = StratifiedKFold(
    n_splits=CV_FOLDS,
    shuffle=True,
    random_state=CV_SEED,
)


CV_SPLITS = []


assignment_rows = []


for fold, (
    train_index,
    dev_index
) in enumerate(
    skf.split(
        episode_table[
            "Episode_ID"
        ],
        episode_table[
            "CV_Stratum"
        ]
    ),
    start=1
):

    train_episodes = set(
        episode_table.iloc[
            train_index
        ][
            "Episode_ID"
        ]
    )


    dev_episodes = set(
        episode_table.iloc[
            dev_index
        ][
            "Episode_ID"
        ]
    )


    overlap = (
        train_episodes
        &
        dev_episodes
    )


    if overlap:

        raise RuntimeError(
            f"Fold {fold}: episode overlap."
        )


    CV_SPLITS.append(
        {
            "fold":
                fold,

            "train_episodes":
                train_episodes,

            "dev_episodes":
                dev_episodes,
        }
    )


    for episode_id in dev_episodes:

        assignment_rows.append(
            {
                "Fold":
                    fold,

                "Episode_ID":
                    episode_id,
            }
        )


    print(
        f"Fold {fold}: "
        f"TRAIN episodes={len(train_episodes)}, "
        f"DEV episodes={len(dev_episodes)}, "
        f"overlap={len(overlap)}"
    )


assignments = pd.DataFrame(
    assignment_rows
)


assignments = assignments.sort_values(
    [
        "Fold",
        "Episode_ID"
    ]
).reset_index(
    drop=True
)


assignments.to_csv(
    HPO_DIR
    / "V2_CV_EPISODE_ASSIGNMENTS.csv",
    index=False
)


# ==================================================================================================
# CHECK IMPORTANT MARGINAL BALANCE
# ==================================================================================================

episode_fold_table = episode_table.merge(
    assignments,
    on="Episode_ID",
    how="left",
    validate="one_to_one"
)


for dimension in [
    "Episode_Type",
    "Episode_Attack_Family",
    "Calibration_Source",
    "Precursor_Regime",
]:

    audit = pd.crosstab(
        episode_fold_table[
            dimension
        ],
        episode_fold_table[
            "Fold"
        ]
    )


    audit.to_csv(
        HPO_DIR
        / f"CV_BALANCE_{dimension}.csv"
    )


    print(
        f"\n{dimension} by DEV fold:"
    )

    print(
        audit.to_string()
    )


# ==================================================================================================
# CONTRACT HASH
#
# Captures:
#   protocol
#   exact fold assignments
#   representation feature vocabulary
#   V2 design
# ==================================================================================================

contract_material = (
    protocol_hash
    +
    "\n"
    +
    assignments.to_csv(
        index=False
    )
    +
    "\n"
    +
    json.dumps(
        REPRESENTATION_NAMES,
        sort_keys=True
    )
    +
    "\n"
    +
    json.dumps(
        {
            "CV_SEED":
                CV_SEED,

            "CV_FOLDS":
                CV_FOLDS,

            "Representations":
                REPRESENTATIONS,

            "Stratification":
                STRAT_COLUMNS,
        },
        sort_keys=True
    )
)


CONTRACT_HASH = hashlib.sha256(
    contract_material.encode(
        "utf-8"
    )
).hexdigest()


print(
    "\nV2 contract hash:"
)

print(
    CONTRACT_HASH
)


# ==================================================================================================
# SAVE CONTRACT
# ==================================================================================================

write_json(
    {
        "Experiment":
            "XGBOOST_W10_TEMPORAL_V2",

        "Representations": {
            "SUMMARY98":
                98,

            "FLAT140":
                140,

            "HYBRID238":
                238,

            "HYBRID_DIFF364":
                364,
        },

        "Representation_Selection":
            "TRAIN_INTERNAL_CV_ONLY",

        "CV":
            "3_fold_episode_level",

        "Stratification_Columns":
            STRAT_COLUMNS,

        "Headline_Metric":
            "Average_Precision",

        "Official_Validation_Used_For_Representation_Selection":
            False,

        "Official_Validation_Used_For_HPO":
            False,

        "TEST_Path_Defined":
            False,

        "TEST_Loaded":
            False,

        "TEST_Used":
            False,

        "Protocol_SHA256":
            protocol_hash,

        "V2_Contract_SHA256":
            CONTRACT_HASH,
    },

    V2_ROOT
    / "XGBOOST_V2_CONTRACT.json"
)


# ==================================================================================================
# CLEAN MEMORY
# ==================================================================================================

del X_LOG
del SUMMARY98
del FLAT140
del DIFF126
del HYBRID238
del HYBRID_DIFF364

gc.collect()


print("\n" + "=" * 120)
print("✓ CELL 5A COMPLETE")
print("=" * 120)


print(
    "SUMMARY98 built ✓"
)

print(
    "FLAT140 exact temporal ordering built ✓"
)

print(
    "HYBRID238 built ✓"
)

print(
    "HYBRID_DIFF364 built ✓"
)

print(
    "Precursor-aware episode CV prepared ✓"
)

print(
    "Official validation used: NO ✓"
)

print(
    "TEST path defined: NO ✓"
)

print(
    "\nRun CELL 5B next."
)

In [ ]:
# ==================================================================================================
# XGBOOST — CELL 5B
#
# SERIOUS V2 OPTIMIZATION
#
# 4 REPRESENTATIONS
# × 12 TRIALS PER REPRESENTATION
# × H5/H10
# × 3 EPISODE FOLDS
#
# REPRESENTATION + HYPERPARAMETER SELECTION IS TRAIN-CV ONLY.
#
# OFFICIAL VALIDATION IS NOT LOADED.
# TEST IS NOT DEFINED.
# ==================================================================================================

from pathlib import Path
import json
import hashlib
import shutil
import gc
import subprocess
import sys

import numpy as np
import pandas as pd

import xgboost as xgb
import optuna

from sklearn.metrics import (
    average_precision_score,
    precision_recall_curve,
    f1_score,
    matthews_corrcoef,
)


optuna.logging.set_verbosity(
    optuna.logging.WARNING
)


print("=" * 120)
print("XGBOOST V2 — CELL 5B")
print("SERIOUS REPRESENTATION + HYPERPARAMETER OPTIMIZATION")
print("=" * 120)


# ==================================================================================================
# PATHS
# ==================================================================================================

ROOT = Path(
    "/content/drive/MyDrive/network-attack-forecasting/Synthetic_NAF_v1"
)

P12_0 = (
    ROOT
    / "model_ready"
    / "12_0"
)

V2_ROOT = (
    ROOT
    / "models"
    / "XGBOOST_W10_TEMPORAL_V2"
)

REP_DIR = (
    V2_ROOT
    / "representations"
)

HPO_DIR = (
    V2_ROOT
    / "hpo"
)


CONTRACT_FILE = (
    V2_ROOT
    / "XGBOOST_V2_CONTRACT.json"
)


TRAIN_Y_FILE = (
    P12_0
    / "W10"
    / "train"
    / "y_binary.npy"
)

TRAIN_ELIGIBLE_FILE = (
    P12_0
    / "W10"
    / "train"
    / "eligible.npy"
)

TRAIN_INDEX_FILE = (
    P12_0
    / "W10"
    / "train"
    / "index.parquet"
)

ASSIGNMENT_FILE = (
    HPO_DIR
    / "V2_CV_EPISODE_ASSIGNMENTS.csv"
)


# ==================================================================================================
# CONSTANTS
# ==================================================================================================

REPRESENTATIONS = [
    "SUMMARY98",
    "FLAT140",
    "HYBRID238",
    "HYBRID_DIFF364",
]


HORIZONS = [
    5,
    10,
]


HORIZON_TO_COLUMN = {
    1: 0,
    5: 1,
    10: 2,
    15: 3,
    30: 4,
}


TRIALS_PER_REPRESENTATION = 12

MAX_BOOST_ROUNDS = 3000

EARLY_STOPPING_ROUNDS = 125

HPO_SEED = 20261023

AP_TIE_TOLERANCE = 0.002


# ==================================================================================================
# REQUIRED FILES
# ==================================================================================================

for path in [
    CONTRACT_FILE,
    TRAIN_Y_FILE,
    TRAIN_ELIGIBLE_FILE,
    TRAIN_INDEX_FILE,
    ASSIGNMENT_FILE,
]:

    if not path.is_file():

        raise FileNotFoundError(
            f"Missing:\n{path}\nRun Cell 5A first."
        )


for representation in REPRESENTATIONS:

    rep_file = (
        REP_DIR
        / f"TRAIN_{representation}.npy"
    )


    names_file = (
        REP_DIR
        / f"{representation}_FEATURE_NAMES.json"
    )


    if not rep_file.is_file():

        raise FileNotFoundError(
            rep_file
        )


    if not names_file.is_file():

        raise FileNotFoundError(
            names_file
        )


# ==================================================================================================
# CONTRACT
# ==================================================================================================

with open(
    CONTRACT_FILE,
    "r",
    encoding="utf-8"
) as f:

    CONTRACT = json.load(
        f
    )


CONTRACT_HASH = CONTRACT[
    "V2_Contract_SHA256"
]


print(
    "Contract hash:",
    CONTRACT_HASH
)


# ==================================================================================================
# DEVICE
# ==================================================================================================

GPU_AVAILABLE = (
    shutil.which(
        "nvidia-smi"
    )
    is not None
)


print(
    "GPU available:",
    GPU_AVAILABLE
)


if GPU_AVAILABLE:

    try:

        gpu_name = subprocess.check_output(
            [
                "nvidia-smi",
                "--query-gpu=name",
                "--format=csv,noheader",
            ],
            text=True
        ).strip()

        print(
            "GPU:",
            gpu_name
        )

    except Exception:

        pass


else:

    print(
        "\nWARNING: Running serious V2 HPO on CPU."
    )

    print(
        "It will still be valid, but enable a T4 if available."
    )


def device_parameters():

    if GPU_AVAILABLE:

        return {
            "tree_method":
                "hist",

            "device":
                "cuda",
        }


    return {
        "tree_method":
            "hist",

        "device":
            "cpu",

        "nthread":
            -1,
    }


# ==================================================================================================
# LOAD LABEL/METADATA
# ==================================================================================================

Y_ALL = np.load(
    TRAIN_Y_FILE,
    allow_pickle=False
)

ELIGIBLE_ALL = np.load(
    TRAIN_ELIGIBLE_FILE,
    allow_pickle=False
)

INDEX = pd.read_parquet(
    TRAIN_INDEX_FILE
)


INDEX = INDEX.copy()


INDEX[
    "Episode_ID"
] = INDEX[
    "Episode_ID"
].astype(
    str
)


EPISODE_IDS = INDEX[
    "Episode_ID"
].to_numpy()


assignments = pd.read_csv(
    ASSIGNMENT_FILE,
    dtype={
        "Episode_ID":
            str
    }
)


# ==================================================================================================
# RECONSTRUCT FIXED CV SPLITS
# ==================================================================================================

all_episodes = set(
    INDEX[
        "Episode_ID"
    ].unique()
)


CV_SPLITS = []


for fold in sorted(
    assignments[
        "Fold"
    ].unique()
):

    dev_episodes = set(
        assignments.loc[
            assignments[
                "Fold"
            ]
            ==
            fold,
            "Episode_ID"
        ]
    )


    train_episodes = (
        all_episodes
        -
        dev_episodes
    )


    if (
        train_episodes
        &
        dev_episodes
    ):

        raise RuntimeError(
            f"Fold {fold}: overlap."
        )


    CV_SPLITS.append(
        {
            "fold":
                int(
                    fold
                ),

            "train_episodes":
                train_episodes,

            "dev_episodes":
                dev_episodes,
        }
    )


# ==================================================================================================
# HELPERS
# ==================================================================================================

def load_representation(
    representation
):

    X = np.load(
        REP_DIR
        / f"TRAIN_{representation}.npy",
        mmap_mode="r"
    )


    with open(
        REP_DIR
        / f"{representation}_FEATURE_NAMES.json",
        "r",
        encoding="utf-8"
    ) as f:

        names = json.load(
            f
        )


    if X.shape[
        1
    ] != len(
        names
    ):

        raise RuntimeError(
            "Feature-name mismatch."
        )


    return (
        X,
        names
    )


def choose_f1_threshold(
    y,
    probability
):

    precision, recall, thresholds = (
        precision_recall_curve(
            y,
            probability
        )
    )


    if len(
        thresholds
    ) == 0:

        return 0.5


    precision = precision[
        :-1
    ]

    recall = recall[
        :-1
    ]


    denom = (
        precision
        +
        recall
    )


    f1 = np.divide(
        2
        *
        precision
        *
        recall,
        denom,
        out=np.zeros_like(
            denom
        ),
        where=(
            denom
            >
            0
        )
    )


    best = f1.max()


    candidates = np.flatnonzero(
        np.isclose(
            f1,
            best,
            atol=1e-12,
            rtol=0.0
        )
    )


    candidate_table = pd.DataFrame(
        {
            "Index":
                candidates,

            "Precision":
                precision[
                    candidates
                ],

            "Threshold":
                thresholds[
                    candidates
                ],
        }
    )


    candidate_table = candidate_table.sort_values(
        [
            "Precision",
            "Threshold"
        ],
        ascending=[
            False,
            False
        ]
    )


    best_index = int(
        candidate_table.iloc[
            0
        ][
            "Index"
        ]
    )


    return float(
        thresholds[
            best_index
        ]
    )


def calculate_pos_weight(
    y,
    power
):

    positive = float(
        np.sum(
            y
            ==
            1
        )
    )


    negative = float(
        np.sum(
            y
            ==
            0
        )
    )


    ratio = (
        negative
        /
        positive
    )


    return float(
        ratio
        **
        power
    )


# ==================================================================================================
# SEARCH SPACE
# ==================================================================================================

def suggest_config(
    trial
):

    grow_policy = trial.suggest_categorical(
        "grow_policy",
        [
            "depthwise",
            "lossguide",
        ]
    )


    if grow_policy == "depthwise":

        max_depth = trial.suggest_int(
            "max_depth",
            3,
            10
        )

        max_leaves = 0


    else:

        max_depth = 0

        max_leaves = trial.suggest_categorical(
            "max_leaves",
            [
                16,
                32,
                64,
                128,
                256,
            ]
        )


    return {
        "grow_policy":
            grow_policy,

        "max_depth":
            max_depth,

        "max_leaves":
            max_leaves,

        "min_child_weight":
            trial.suggest_float(
                "min_child_weight",
                0.5,
                30.0,
                log=True
            ),

        "learning_rate":
            trial.suggest_float(
                "learning_rate",
                0.01,
                0.15,
                log=True
            ),

        "subsample":
            trial.suggest_float(
                "subsample",
                0.60,
                1.0
            ),

        "colsample_bytree":
            trial.suggest_float(
                "colsample_bytree",
                0.50,
                1.0
            ),

        "colsample_bylevel":
            trial.suggest_float(
                "colsample_bylevel",
                0.60,
                1.0
            ),

        "gamma":
            trial.suggest_float(
                "gamma",
                0.0,
                4.0
            ),

        "reg_alpha":
            trial.suggest_float(
                "reg_alpha",
                1e-7,
                5.0,
                log=True
            ),

        "reg_lambda":
            trial.suggest_float(
                "reg_lambda",
                1e-3,
                30.0,
                log=True
            ),

        "max_bin":
            trial.suggest_categorical(
                "max_bin",
                [
                    128,
                    256,
                    512,
                ]
            ),

        "max_delta_step":
            trial.suggest_categorical(
                "max_delta_step",
                [
                    0,
                    1,
                    5,
                ]
            ),

        "class_weight_power":
            trial.suggest_categorical(
                "class_weight_power",
                [
                    0.0,
                    0.25,
                    0.50,
                    0.75,
                    1.0,
                ]
            ),
    }


# ==================================================================================================
# TRAIN ONE FOLD
# ==================================================================================================

def train_one_fold(
    *,
    X_ALL,
    feature_names,
    H,
    fold,
    config,
):

    h_col = HORIZON_TO_COLUMN[
        H
    ]


    eligible = (
        ELIGIBLE_ALL[
            :,
            h_col
        ]
        ==
        1
    )


    train_mask = (

        eligible

        &

        np.isin(
            EPISODE_IDS,
            list(
                fold[
                    "train_episodes"
                ]
            )
        )
    )


    dev_mask = (

        eligible

        &

        np.isin(
            EPISODE_IDS,
            list(
                fold[
                    "dev_episodes"
                ]
            )
        )
    )


    if np.any(
        train_mask
        &
        dev_mask
    ):

        raise RuntimeError(
            "CV leakage."
        )


    X_train = np.asarray(
        X_ALL[
            train_mask
        ],
        dtype=np.float32
    )


    X_dev = np.asarray(
        X_ALL[
            dev_mask
        ],
        dtype=np.float32
    )


    y_train = np.asarray(
        Y_ALL[
            train_mask,
            h_col
        ],
        dtype=np.int8
    )


    y_dev = np.asarray(
        Y_ALL[
            dev_mask,
            h_col
        ],
        dtype=np.int8
    )


    scale_pos_weight = calculate_pos_weight(
        y_train,
        config[
            "class_weight_power"
        ]
    )


    dtrain = xgb.DMatrix(
        X_train,
        label=y_train,
        feature_names=feature_names
    )


    ddev = xgb.DMatrix(
        X_dev,
        label=y_dev,
        feature_names=feature_names
    )


    params = {
        "objective":
            "binary:logistic",

        "eval_metric":
            "aucpr",

        "grow_policy":
            config[
                "grow_policy"
            ],

        "max_depth":
            config[
                "max_depth"
            ],

        "max_leaves":
            config[
                "max_leaves"
            ],

        "min_child_weight":
            config[
                "min_child_weight"
            ],

        "eta":
            config[
                "learning_rate"
            ],

        "subsample":
            config[
                "subsample"
            ],

        "colsample_bytree":
            config[
                "colsample_bytree"
            ],

        "colsample_bylevel":
            config[
                "colsample_bylevel"
            ],

        "gamma":
            config[
                "gamma"
            ],

        "alpha":
            config[
                "reg_alpha"
            ],

        "lambda":
            config[
                "reg_lambda"
            ],

        "max_bin":
            config[
                "max_bin"
            ],

        "max_delta_step":
            config[
                "max_delta_step"
            ],

        "scale_pos_weight":
            scale_pos_weight,

        "seed":
            (
                HPO_SEED
                +
                H
                *
                100
                +
                fold[
                    "fold"
                ]
            ),

        "verbosity":
            0,

        **device_parameters(),
    }


    model = xgb.train(
        params=params,
        dtrain=dtrain,
        num_boost_round=MAX_BOOST_ROUNDS,
        evals=[
            (
                ddev,
                "dev"
            )
        ],
        early_stopping_rounds=EARLY_STOPPING_ROUNDS,
        verbose_eval=False,
    )


    best_iteration = int(
        model.best_iteration
    )


    probability = model.predict(
        ddev,
        iteration_range=(
            0,
            best_iteration
            +
            1
        )
    )


    ap = float(
        average_precision_score(
            y_dev,
            probability
        )
    )


    threshold = choose_f1_threshold(
        y_dev,
        probability
    )


    prediction = (
        probability
        >=
        threshold
    ).astype(
        np.int8
    )


    f1 = float(
        f1_score(
            y_dev,
            prediction,
            zero_division=0
        )
    )


    mcc = float(
        matthews_corrcoef(
            y_dev,
            prediction
        )
    )


    return {
        "AP":
            ap,

        "F1":
            f1,

        "MCC":
            mcc,

        "Best_Iteration":
            best_iteration,

        "Scale_Pos_Weight":
            scale_pos_weight,
    }


# ==================================================================================================
# OBJECTIVE FACTORY
# ==================================================================================================

def make_objective(
    representation,
    H,
    X_ALL,
    feature_names,
):

    def objective(
        trial
    ):

        config = suggest_config(
            trial
        )


        fold_results = []


        for fold in CV_SPLITS:

            result = train_one_fold(
                X_ALL=X_ALL,
                feature_names=feature_names,
                H=H,
                fold=fold,
                config=config,
            )


            fold_results.append(
                result
            )


            running_ap = float(
                np.mean(
                    [
                        row[
                            "AP"
                        ]
                        for row
                        in fold_results
                    ]
                )
            )


            print(
                f"H{H} | "
                f"{representation:<15} | "
                f"trial={trial.number:02d} | "
                f"fold={fold['fold']} | "
                f"AP={result['AP']:.6f} | "
                f"running={running_ap:.6f}"
            )


            trial.report(
                running_ap,
                step=fold[
                    "fold"
                ]
            )


            if (
                fold[
                    "fold"
                ]
                >=
                2

                and

                trial.should_prune()
            ):

                raise optuna.TrialPruned()


        aps = np.array(
            [
                row[
                    "AP"
                ]
                for row
                in fold_results
            ],
            dtype=float
        )


        f1s = np.array(
            [
                row[
                    "F1"
                ]
                for row
                in fold_results
            ],
            dtype=float
        )


        mccs = np.array(
            [
                row[
                    "MCC"
                ]
                for row
                in fold_results
            ],
            dtype=float
        )


        iterations = np.array(
            [
                row[
                    "Best_Iteration"
                ]
                for row
                in fold_results
            ],
            dtype=float
        )


        trial.set_user_attr(
            "V2_contract_hash",
            CONTRACT_HASH
        )


        trial.set_user_attr(
            "representation",
            representation
        )


        trial.set_user_attr(
            "resolved_config",
            config
        )


        trial.set_user_attr(
            "fold_aps",
            aps.tolist()
        )


        trial.set_user_attr(
            "mean_ap",
            float(
                aps.mean()
            )
        )


        trial.set_user_attr(
            "std_ap",
            float(
                aps.std(
                    ddof=1
                )
            )
        )


        trial.set_user_attr(
            "min_fold_ap",
            float(
                aps.min()
            )
        )


        trial.set_user_attr(
            "mean_f1",
            float(
                f1s.mean()
            )
        )


        trial.set_user_attr(
            "mean_mcc",
            float(
                mccs.mean()
            )
        )


        trial.set_user_attr(
            "mean_best_iteration",
            float(
                iterations.mean()
            )
        )


        print(
            f"{representation} H{H} "
            f"TRIAL {trial.number} COMPLETE | "
            f"AP={aps.mean():.6f} ± {aps.std(ddof=1):.6f} | "
            f"worst={aps.min():.6f}"
        )


        return float(
            aps.mean()
        )


    return objective


# ==================================================================================================
# ANCHORS
#
# Two deliberate starting points per representation.
# ==================================================================================================

ANCHORS = [

    {
        "grow_policy":
            "depthwise",

        "max_depth":
            4,

        "min_child_weight":
            3.0,

        "learning_rate":
            0.05,

        "subsample":
            0.85,

        "colsample_bytree":
            0.85,

        "colsample_bylevel":
            0.90,

        "gamma":
            0.0,

        "reg_alpha":
            0.001,

        "reg_lambda":
            1.0,

        "max_bin":
            256,

        "max_delta_step":
            0,

        "class_weight_power":
            0.0,
    },

    {
        "grow_policy":
            "depthwise",

        "max_depth":
            7,

        "min_child_weight":
            8.0,

        "learning_rate":
            0.025,

        "subsample":
            0.80,

        "colsample_bytree":
            0.75,

        "colsample_bylevel":
            0.85,

        "gamma":
            0.5,

        "reg_alpha":
            0.05,

        "reg_lambda":
            3.0,

        "max_bin":
            512,

        "max_delta_step":
            1,

        "class_weight_power":
            0.5,
    },
]


# ==================================================================================================
# RUN ALL REPRESENTATIONS
# ==================================================================================================

representation_winner_rows = []


for H in HORIZONS:

    print("\n" + "#" * 120)

    print(
        f"XGBOOST V2 HORIZON H{H}"
    )

    print("#" * 120)


    horizon_trial_candidates = []


    for representation in REPRESENTATIONS:

        print("\n" + "=" * 120)

        print(
            f"H{H} — {representation}"
        )

        print("=" * 120)


        X_REP, FEATURE_NAMES = load_representation(
            representation
        )


        study_name = (
            f"XGB_V2_H{H}_{representation}_"
            +
            CONTRACT_HASH[
                :12
            ]
        )


        storage_file = (
            HPO_DIR
            / "xgboost_v2_optuna.sqlite3"
        )


        study = optuna.create_study(
            study_name=study_name,
            storage=(
                "sqlite:///"
                +
                str(
                    storage_file
                )
            ),
            direction="maximize",
            sampler=optuna.samplers.TPESampler(
                seed=(
                    HPO_SEED
                    +
                    H
                    +
                    REPRESENTATIONS.index(
                        representation
                    )
                    *
                    1000
                )
            ),
            pruner=optuna.pruners.MedianPruner(
                n_startup_trials=5,
                n_warmup_steps=2,
            ),
            load_if_exists=True,
        )


        # ------------------------------------------------------------------------------------------
        # CONTRACT SAFETY
        # ------------------------------------------------------------------------------------------

        for old_trial in study.trials:

            if (
                old_trial.state
                ==
                optuna.trial.TrialState.COMPLETE
            ):

                old_hash = old_trial.user_attrs.get(
                    "V2_contract_hash"
                )


                if old_hash != CONTRACT_HASH:

                    raise RuntimeError(
                        "Optuna study contract mismatch."
                    )


        if len(
            study.trials
        ) == 0:

            for anchor in ANCHORS:

                study.enqueue_trial(
                    anchor
                )


        finished = sum(
            trial.state
            in {
                optuna.trial.TrialState.COMPLETE,
                optuna.trial.TrialState.PRUNED,
                optuna.trial.TrialState.FAIL,
            }

            for trial
            in study.trials
        )


        remaining = max(
            0,
            TRIALS_PER_REPRESENTATION
            -
            finished
        )


        print(
            "Existing finished:",
            finished
        )


        print(
            "Remaining:",
            remaining
        )


        if remaining > 0:

            study.optimize(
                make_objective(
                    representation,
                    H,
                    X_REP,
                    FEATURE_NAMES,
                ),
                n_trials=remaining,
                gc_after_trial=True,
            )


        completed = [
            trial

            for trial
            in study.trials

            if trial.state
            ==
            optuna.trial.TrialState.COMPLETE
        ]


        if not completed:

            raise RuntimeError(
                f"No completed trials for {representation} H{H}."
            )


        study.trials_dataframe().to_csv(
            HPO_DIR
            / f"H{H:02d}_{representation}_TRIALS.csv",
            index=False
        )


        best_mean_ap = max(
            float(
                trial.value
            )

            for trial
            in completed
        )


        # ==========================================================================================
        # PRIMARY METRIC-FIRST SELECTION
        #
        # Any candidate within 0.002 AP of maximum:
        #
        #   1. lower fold-to-fold AP std
        #   2. higher worst-fold AP
        #   3. fewer boosting iterations
        #   4. lower trial number
        #
        # NO F1 TIEBREAK.
        # ==========================================================================================

        candidates = [
            trial

            for trial
            in completed

            if (
                best_mean_ap
                -
                float(
                    trial.value
                )
            )
            <=
            AP_TIE_TOLERANCE
        ]


        candidates = sorted(
            candidates,
            key=lambda trial: (
                float(
                    trial.user_attrs[
                        "std_ap"
                    ]
                ),
                -float(
                    trial.user_attrs[
                        "min_fold_ap"
                    ]
                ),
                float(
                    trial.user_attrs[
                        "mean_best_iteration"
                    ]
                ),
                trial.number,
            )
        )


        selected = candidates[
            0
        ]


        record = {
            "Horizon":
                H,

            "Representation":
                representation,

            "Trial":
                int(
                    selected.number
                ),

            "Mean_CV_AP":
                float(
                    selected.value
                ),

            "CV_AP_Std":
                float(
                    selected.user_attrs[
                        "std_ap"
                    ]
                ),

            "Worst_Fold_AP":
                float(
                    selected.user_attrs[
                        "min_fold_ap"
                    ]
                ),

            "Mean_CV_F1":
                float(
                    selected.user_attrs[
                        "mean_f1"
                    ]
                ),

            "Mean_CV_MCC":
                float(
                    selected.user_attrs[
                        "mean_mcc"
                    ]
                ),

            "Mean_Best_Iteration":
                float(
                    selected.user_attrs[
                        "mean_best_iteration"
                    ]
                ),

            "Configuration":
                selected.user_attrs[
                    "resolved_config"
                ],

            "Contract_Hash":
                CONTRACT_HASH,
        }


        with open(
            HPO_DIR
            / f"H{H:02d}_{representation}_SELECTED.json",
            "w",
            encoding="utf-8"
        ) as f:

            json.dump(
                record,
                f,
                indent=4
            )


        representation_winner_rows.append(
            {
                key:
                    value

                for key, value
                in record.items()

                if key
                !=
                "Configuration"
            }
        )


        horizon_trial_candidates.append(
            record
        )


        print(
            "\nREPRESENTATION WINNER:"
        )


        print(
            json.dumps(
                record,
                indent=4
            )
        )


        del X_REP

        gc.collect()


    # ==============================================================================================
    # CHOOSE REPRESENTATION FOR THIS HORIZON
    # ==============================================================================================

    best_rep_ap = max(
        row[
            "Mean_CV_AP"
        ]

        for row
        in horizon_trial_candidates
    )


    representation_candidates = [
        row

        for row
        in horizon_trial_candidates

        if (
            best_rep_ap
            -
            row[
                "Mean_CV_AP"
            ]
        )
        <=
        AP_TIE_TOLERANCE
    ]


    representation_candidates = sorted(
        representation_candidates,
        key=lambda row: (
            row[
                "CV_AP_Std"
            ],
            -row[
                "Worst_Fold_AP"
            ],
            row[
                "Mean_Best_Iteration"
            ],
            REPRESENTATIONS.index(
                row[
                    "Representation"
                ]
            ),
        )
    )


    selected_horizon = representation_candidates[
        0
    ]


    with open(
        HPO_DIR
        / f"H{H:02d}_FINAL_V2_SELECTION.json",
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            selected_horizon,
            f,
            indent=4
        )


    print("\n" + "#" * 120)

    print(
        f"H{H} FINAL TRAIN-CV SELECTION"
    )

    print("#" * 120)


    print(
        json.dumps(
            selected_horizon,
            indent=4
        )
    )


# ==================================================================================================
# SUMMARY TABLE
# ==================================================================================================

representation_summary = pd.DataFrame(
    representation_winner_rows
)


representation_summary.to_csv(
    HPO_DIR
    / "V2_REPRESENTATION_HPO_SUMMARY.csv",
    index=False
)


print("\n" + "=" * 120)
print("V2 REPRESENTATION HPO SUMMARY")
print("=" * 120)


print(
    representation_summary[
        [
            "Horizon",
            "Representation",
            "Trial",
            "Mean_CV_AP",
            "CV_AP_Std",
            "Worst_Fold_AP",
            "Mean_CV_F1",
            "Mean_CV_MCC",
            "Mean_Best_Iteration",
        ]
    ].to_string(
        index=False
    )
)


# ==================================================================================================
# COMPLETE
# ==================================================================================================

with open(
    HPO_DIR
    / "CELL5B_V2_HPO_COMPLETE.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        {
            "Status":
                "COMPLETE",

            "Representations":
                REPRESENTATIONS,

            "Trials_Per_Representation":
                TRIALS_PER_REPRESENTATION,

            "Horizons":
                HORIZONS,

            "Selection":
                "TRAIN_EPISODE_CV_AP",

            "Official_Validation_Loaded":
                False,

            "Official_Validation_Used":
                False,

            "TEST_Path_Defined":
                False,

            "TEST_Loaded":
                False,

            "TEST_Used":
                False,

            "Contract_Hash":
                CONTRACT_HASH,
        },
        f,
        indent=4
    )


print("\n" + "=" * 120)
print("✓ CELL 5B COMPLETE")
print("=" * 120)

print(
    "Representation selection: TRAIN CV only ✓"
)

print(
    "Hyperparameter selection: TRAIN CV only ✓"
)

print(
    "Official validation loaded: NO ✓"
)

print(
    "TEST loaded: NO ✓"
)

print(
    "\nRun CELL 5C next."
)

In [ ]:
# ==================================================================================================
# XGBOOST — CELL 5C
#
# FINAL V2 CONFIRMATION
#
# Uses:
#   representation chosen by TRAIN-CV
#   hyperparameters chosen by TRAIN-CV
#
# Official VALIDATION is now used ONCE for:
#   early stopping
#   F1 threshold
#   final development metrics
#
# Compares:
#
#   XGBOOST V1
#   XGBOOST V2
#   ORIGINAL LSTM
#
# TEST IS NOT DEFINED.
# ==================================================================================================

from pathlib import Path
import json
import shutil
import gc

import numpy as np
import pandas as pd

import xgboost as xgb

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    accuracy_score,
    balanced_accuracy_score,
    matthews_corrcoef,
    brier_score_loss,
    confusion_matrix,
    precision_recall_curve,
    auc,
)


print("=" * 120)
print("XGBOOST V2 — CELL 5C")
print("FINAL TRAIN → OFFICIAL VALIDATION CONFIRMATION")
print("=" * 120)


# ==================================================================================================
# PATHS
# ==================================================================================================

ROOT = Path(
    "/content/drive/MyDrive/network-attack-forecasting/Synthetic_NAF_v1"
)

P12_0 = (
    ROOT
    / "model_ready"
    / "12_0"
)

V1_ROOT = (
    ROOT
    / "models"
    / "XGBOOST_W10_TEMPORAL_SUMMARY"
)

V2_ROOT = (
    ROOT
    / "models"
    / "XGBOOST_W10_TEMPORAL_V2"
)

REP_DIR = (
    V2_ROOT
    / "representations"
)

HPO_DIR = (
    V2_ROOT
    / "hpo"
)

FINAL_DIR = (
    V2_ROOT
    / "final_validation"
)

FINAL_DIR.mkdir(
    parents=True,
    exist_ok=True
)


TRAIN_RAW_FILE = (
    P12_0
    / "W10"
    / "train"
    / "X.npy"
)

VAL_RAW_FILE = (
    P12_0
    / "W10"
    / "validation"
    / "X.npy"
)


TRAIN_Y_FILE = (
    P12_0
    / "W10"
    / "train"
    / "y_binary.npy"
)

TRAIN_ELIGIBLE_FILE = (
    P12_0
    / "W10"
    / "train"
    / "eligible.npy"
)


VAL_Y_FILE = (
    P12_0
    / "W10"
    / "validation"
    / "y_binary.npy"
)

VAL_ELIGIBLE_FILE = (
    P12_0
    / "W10"
    / "validation"
    / "eligible.npy"
)

VAL_INDEX_FILE = (
    P12_0
    / "W10"
    / "validation"
    / "index.parquet"
)


# ==================================================================================================
# CONSTANTS
# ==================================================================================================

HORIZONS = [
    5,
    10,
]


HORIZON_TO_COLUMN = {
    1: 0,
    5: 1,
    10: 2,
    15: 3,
    30: 4,
}


FINAL_SEEDS = [
    20260930,
    20260931,
    20260932,
]


WINDOW = 10


BASE_FEATURES = [
    "Flow_Count",
    "Flow_Duration_Mean",
    "Flow_Duration_Std",
    "Flow_Duration_Max",
    "Forward_Packets_Sum",
    "Backward_Packets_Sum",
    "Total_Packets",
    "Forward_Bytes_Sum",
    "Backward_Bytes_Sum",
    "Total_Bytes",
    "Packets_Per_Second",
    "Bytes_Per_Second",
    "Forward_Backward_Packet_Ratio",
    "Forward_Backward_Byte_Ratio",
]


# ==================================================================================================
# DEVICE
# ==================================================================================================

GPU_AVAILABLE = (
    shutil.which(
        "nvidia-smi"
    )
    is not None
)


def device_parameters():

    if GPU_AVAILABLE:

        return {
            "tree_method":
                "hist",

            "device":
                "cuda",
        }


    return {
        "tree_method":
            "hist",

        "device":
            "cpu",

        "nthread":
            -1,
    }


print(
    "GPU available:",
    GPU_AVAILABLE
)


# ==================================================================================================
# TEMPORAL REPRESENTATION FUNCTION
# ==================================================================================================

def build_representation(
    X_raw,
    representation
):

    X = np.log1p(
        np.asarray(
            X_raw,
            dtype=np.float32
        )
    )


    last = X[
        :,
        -1,
        :
    ]


    mean = X.mean(
        axis=1
    )


    std = X.std(
        axis=1,
        ddof=0
    )


    minimum = X.min(
        axis=1
    )


    maximum = X.max(
        axis=1
    )


    delta = (
        X[
            :,
            -1,
            :
        ]
        -
        X[
            :,
            0,
            :
        ]
    )


    t = np.arange(
        WINDOW,
        dtype=np.float32
    )


    tc = (
        t
        -
        t.mean()
    )


    slope = (

        np.sum(
            X
            *
            tc[
                None,
                :,
                None
            ],
            axis=1
        )

        /

        np.sum(
            tc ** 2
        )
    )


    summary = np.concatenate(
        [
            last,
            mean,
            std,
            minimum,
            maximum,
            delta,
            slope,
        ],
        axis=1
    ).astype(
        np.float32
    )


    flat = X.reshape(
        len(
            X
        ),
        -1
    ).astype(
        np.float32
    )


    diff = np.diff(
        X,
        axis=1
    ).reshape(
        len(
            X
        ),
        -1
    ).astype(
        np.float32
    )


    if representation == "SUMMARY98":

        return summary


    if representation == "FLAT140":

        return flat


    if representation == "HYBRID238":

        return np.concatenate(
            [
                flat,
                summary,
            ],
            axis=1
        ).astype(
            np.float32
        )


    if representation == "HYBRID_DIFF364":

        return np.concatenate(
            [
                flat,
                summary,
                diff,
            ],
            axis=1
        ).astype(
            np.float32
        )


    raise ValueError(
        representation
    )


# ==================================================================================================
# HELPERS
# ==================================================================================================

def choose_f1_threshold(
    y,
    probability
):

    precision, recall, thresholds = (
        precision_recall_curve(
            y,
            probability
        )
    )


    if len(
        thresholds
    ) == 0:

        return 0.5


    precision = precision[
        :-1
    ]

    recall = recall[
        :-1
    ]


    denom = (
        precision
        +
        recall
    )


    f1 = np.divide(
        2
        *
        precision
        *
        recall,
        denom,
        out=np.zeros_like(
            denom
        ),
        where=(
            denom
            >
            0
        )
    )


    best_f1 = f1.max()


    candidates = np.flatnonzero(
        np.isclose(
            f1,
            best_f1,
            atol=1e-12,
            rtol=0
        )
    )


    candidate_table = pd.DataFrame(
        {
            "Index":
                candidates,

            "Precision":
                precision[
                    candidates
                ],

            "Threshold":
                thresholds[
                    candidates
                ],
        }
    )


    candidate_table = candidate_table.sort_values(
        [
            "Precision",
            "Threshold"
        ],
        ascending=[
            False,
            False
        ]
    )


    best_index = int(
        candidate_table.iloc[
            0
        ][
            "Index"
        ]
    )


    return float(
        thresholds[
            best_index
        ]
    )


def calculate_metrics(
    y,
    probability,
    threshold
):

    prediction = (
        probability
        >=
        threshold
    ).astype(
        np.int8
    )


    tn, fp, fn, tp = confusion_matrix(
        y,
        prediction,
        labels=[
            0,
            1
        ]
    ).ravel()


    specificity = (
        tn
        /
        (
            tn
            +
            fp
        )
    )


    p_curve, r_curve, _ = precision_recall_curve(
        y,
        probability
    )


    return {
        "Average_Precision":
            float(
                average_precision_score(
                    y,
                    probability
                )
            ),

        "PR_AUC":
            float(
                auc(
                    r_curve,
                    p_curve
                )
            ),

        "ROC_AUC":
            float(
                roc_auc_score(
                    y,
                    probability
                )
            ),

        "Accuracy":
            float(
                accuracy_score(
                    y,
                    prediction
                )
            ),

        "Balanced_Accuracy":
            float(
                balanced_accuracy_score(
                    y,
                    prediction
                )
            ),

        "Precision":
            float(
                precision_score(
                    y,
                    prediction,
                    zero_division=0
                )
            ),

        "Recall":
            float(
                recall_score(
                    y,
                    prediction,
                    zero_division=0
                )
            ),

        "Specificity":
            float(
                specificity
            ),

        "F1":
            float(
                f1_score(
                    y,
                    prediction,
                    zero_division=0
                )
            ),

        "MCC":
            float(
                matthews_corrcoef(
                    y,
                    prediction
                )
            ),

        "Brier":
            float(
                brier_score_loss(
                    y,
                    probability
                )
            ),

        "Threshold":
            float(
                threshold
            ),

        "TN":
            int(
                tn
            ),

        "FP":
            int(
                fp
            ),

        "FN":
            int(
                fn
            ),

        "TP":
            int(
                tp
            ),
    }


def calculate_event_metrics(
    y,
    probability,
    threshold,
    metadata
):

    prediction = (
        probability
        >=
        threshold
    )


    episodes = (
        metadata[
            "Episode_ID"
        ]
        .astype(str)
        .to_numpy()
    )


    tto = pd.to_numeric(
        metadata[
            "Minutes_To_Next_Onset"
        ],
        errors="coerce"
    ).to_numpy(
        dtype=float
    )


    positive = (
        y
        ==
        1
    )


    negative = (
        y
        ==
        0
    )


    tp_warning = (
        prediction
        &
        positive
    )


    fp_warning = (
        prediction
        &
        negative
    )


    true_events = set(
        episodes[
            positive
        ]
    )


    recalled_events = set(
        episodes[
            tp_warning
        ]
    )


    warned_episodes = set(
        episodes[
            prediction
        ]
    )


    event_recall = (
        len(
            recalled_events
        )
        /
        len(
            true_events
        )
    )


    event_precision = (
        len(
            recalled_events
        )
        /
        len(
            warned_episodes
        )

        if len(
            warned_episodes
        )
        >
        0

        else np.nan
    )


    leads = []


    for episode in recalled_events:

        mask = (

            tp_warning

            &

            (
                episodes
                ==
                episode
            )
        )


        values = tto[
            mask
        ]


        values = values[
            np.isfinite(
                values
            )
        ]


        if len(
            values
        ):

            leads.append(
                float(
                    values.max()
                )
            )


    negative_hours = (
        negative.sum()
        /
        60.0
    )


    fp_rate = (
        fp_warning.sum()
        /
        negative_hours
    )


    return {
        "Event_Recall":
            float(
                event_recall
            ),

        "Event_Precision":
            float(
                event_precision
            ),

        "Median_First_Warning_Lead_Min":
            float(
                np.median(
                    leads
                )
            )
            if leads
            else np.nan,

        "Mean_First_Warning_Lead_Min":
            float(
                np.mean(
                    leads
                )
            )
            if leads
            else np.nan,

        "False_Warning_Origins_Per_Hour":
            float(
                fp_rate
            ),
    }


def calculate_pos_weight(
    y,
    power
):

    positive = float(
        (
            y
            ==
            1
        ).sum()
    )


    negative = float(
        (
            y
            ==
            0
        ).sum()
    )


    return float(
        (
            negative
            /
            positive
        )
        **
        power
    )


# ==================================================================================================
# LOAD RAW TRAIN + VALIDATION
# ==================================================================================================

X_TRAIN_RAW = np.load(
    TRAIN_RAW_FILE,
    mmap_mode="r"
)

X_VAL_RAW = np.load(
    VAL_RAW_FILE,
    mmap_mode="r"
)


Y_TRAIN_ALL = np.load(
    TRAIN_Y_FILE,
    allow_pickle=False
)

ELIGIBLE_TRAIN = np.load(
    TRAIN_ELIGIBLE_FILE,
    allow_pickle=False
)


Y_VAL_ALL = np.load(
    VAL_Y_FILE,
    allow_pickle=False
)

ELIGIBLE_VAL = np.load(
    VAL_ELIGIBLE_FILE,
    allow_pickle=False
)


VAL_INDEX = pd.read_parquet(
    VAL_INDEX_FILE
)


VAL_INDEX = VAL_INDEX.copy()


VAL_INDEX[
    "Episode_ID"
] = VAL_INDEX[
    "Episode_ID"
].astype(
    str
)


# ==================================================================================================
# FINAL RUN
# ==================================================================================================

all_rows = []


for H in HORIZONS:

    print("\n" + "#" * 120)

    print(
        f"XGBOOST V2 FINAL DEVELOPMENT — H{H}"
    )

    print("#" * 120)


    selection_file = (
        HPO_DIR
        / f"H{H:02d}_FINAL_V2_SELECTION.json"
    )


    if not selection_file.is_file():

        raise FileNotFoundError(
            f"Run Cell 5B first:\n{selection_file}"
        )


    with open(
        selection_file,
        "r",
        encoding="utf-8"
    ) as f:

        selection = json.load(
            f
        )


    representation = selection[
        "Representation"
    ]


    config = selection[
        "Configuration"
    ]


    print(
        "Selected representation:",
        representation
    )


    print(
        "Selected TRAIN-CV AP:",
        selection[
            "Mean_CV_AP"
        ]
    )


    print(
        "Configuration:"
    )


    print(
        json.dumps(
            config,
            indent=4
        )
    )


    # ==============================================================================================
    # BUILD EXACT SAME REPRESENTATION
    # ==============================================================================================

    X_train_all_rep = build_representation(
        X_TRAIN_RAW,
        representation
    )


    X_val_all_rep = build_representation(
        X_VAL_RAW,
        representation
    )


    with open(
        REP_DIR
        / f"{representation}_FEATURE_NAMES.json",
        "r",
        encoding="utf-8"
    ) as f:

        FEATURE_NAMES = json.load(
            f
        )


    if X_train_all_rep.shape[
        1
    ] != len(
        FEATURE_NAMES
    ):

        raise RuntimeError(
            "Representation feature mismatch."
        )


    h_col = HORIZON_TO_COLUMN[
        H
    ]


    train_mask = (
        ELIGIBLE_TRAIN[
            :,
            h_col
        ]
        ==
        1
    )


    val_mask = (
        ELIGIBLE_VAL[
            :,
            h_col
        ]
        ==
        1
    )


    X_train = np.asarray(
        X_train_all_rep[
            train_mask
        ],
        dtype=np.float32
    )


    X_val = np.asarray(
        X_val_all_rep[
            val_mask
        ],
        dtype=np.float32
    )


    y_train = np.asarray(
        Y_TRAIN_ALL[
            train_mask,
            h_col
        ],
        dtype=np.int8
    )


    y_val = np.asarray(
        Y_VAL_ALL[
            val_mask,
            h_col
        ],
        dtype=np.int8
    )


    val_metadata = (
        VAL_INDEX
        .loc[
            val_mask
        ]
        .reset_index(
            drop=True
        )
    )


    print(
        "\nTRAIN:",
        X_train.shape,
        "positive=",
        int(
            y_train.sum()
        )
    )


    print(
        "VALIDATION:",
        X_val.shape,
        "positive=",
        int(
            y_val.sum()
        )
    )


    for seed in FINAL_SEEDS:

        print("\n" + "-" * 120)

        print(
            f"H{H} | "
            f"{representation} | "
            f"seed={seed}"
        )

        print("-" * 120)


        scale_pos_weight = calculate_pos_weight(
            y_train,
            config[
                "class_weight_power"
            ]
        )


        dtrain = xgb.DMatrix(
            X_train,
            label=y_train,
            feature_names=FEATURE_NAMES
        )


        dval = xgb.DMatrix(
            X_val,
            label=y_val,
            feature_names=FEATURE_NAMES
        )


        params = {
            "objective":
                "binary:logistic",

            "eval_metric":
                "aucpr",

            "grow_policy":
                config[
                    "grow_policy"
                ],

            "max_depth":
                config[
                    "max_depth"
                ],

            "max_leaves":
                config[
                    "max_leaves"
                ],

            "min_child_weight":
                config[
                    "min_child_weight"
                ],

            "eta":
                config[
                    "learning_rate"
                ],

            "subsample":
                config[
                    "subsample"
                ],

            "colsample_bytree":
                config[
                    "colsample_bytree"
                ],

            "colsample_bylevel":
                config[
                    "colsample_bylevel"
                ],

            "gamma":
                config[
                    "gamma"
                ],

            "alpha":
                config[
                    "reg_alpha"
                ],

            "lambda":
                config[
                    "reg_lambda"
                ],

            "max_bin":
                config[
                    "max_bin"
                ],

            "max_delta_step":
                config[
                    "max_delta_step"
                ],

            "scale_pos_weight":
                scale_pos_weight,

            "seed":
                seed,

            "verbosity":
                0,

            **device_parameters(),
        }


        history = {}


        model = xgb.train(
            params=params,
            dtrain=dtrain,
            num_boost_round=3500,
            evals=[
                (
                    dval,
                    "validation"
                )
            ],
            early_stopping_rounds=150,
            evals_result=history,
            verbose_eval=False,
        )


        best_iteration = int(
            model.best_iteration
        )


        probability = model.predict(
            dval,
            iteration_range=(
                0,
                best_iteration
                +
                1
            )
        )


        threshold = choose_f1_threshold(
            y_val,
            probability
        )


        metrics = calculate_metrics(
            y_val,
            probability,
            threshold
        )


        event_metrics = calculate_event_metrics(
            y_val,
            probability,
            threshold,
            val_metadata
        )


        record = {
            "Model":
                "XGBOOST_W10_TEMPORAL_V2",

            "Horizon":
                H,

            "Representation":
                representation,

            "Seed":
                seed,

            "Best_Iteration":
                best_iteration,

            "Scale_Pos_Weight":
                scale_pos_weight,

            **config,

            **metrics,

            **event_metrics,

            "TEST_Used":
                False,
        }


        all_rows.append(
            record
        )


        seed_dir = (
            FINAL_DIR
            / f"H{H:02d}"
            / f"seed_{seed}"
        )


        seed_dir.mkdir(
            parents=True,
            exist_ok=True
        )


        model.save_model(
            seed_dir
            / "model.json"
        )


        with open(
            seed_dir
            / "validation_metrics.json",
            "w",
            encoding="utf-8"
        ) as f:

            json.dump(
                record,
                f,
                indent=4
            )


        predictions = val_metadata[
            [
                "Episode_ID",
                "Minute_Index",
                "Minutes_To_Next_Onset",
            ]
        ].copy()


        predictions[
            "Y_True"
        ] = y_val


        predictions[
            "Probability"
        ] = probability


        predictions[
            "Threshold"
        ] = threshold


        predictions[
            "Prediction"
        ] = (
            probability
            >=
            threshold
        ).astype(
            np.int8
        )


        predictions.to_parquet(
            seed_dir
            / "validation_predictions.parquet",
            index=False
        )


        importance_gain = model.get_score(
            importance_type="gain"
        )


        importance = pd.DataFrame(
            {
                "Feature":
                    FEATURE_NAMES,

                "Gain":
                    [
                        importance_gain.get(
                            feature,
                            0.0
                        )

                        for feature
                        in FEATURE_NAMES
                    ]
            }
        ).sort_values(
            "Gain",
            ascending=False
        )


        importance.to_csv(
            seed_dir
            / "feature_importance_gain.csv",
            index=False
        )


        print(
            f"AP             = {metrics['Average_Precision']:.6f}"
        )

        print(
            f"PR-AUC         = {metrics['PR_AUC']:.6f}"
        )

        print(
            f"ROC-AUC        = {metrics['ROC_AUC']:.6f}"
        )

        print(
            f"Precision      = {metrics['Precision']:.6f}"
        )

        print(
            f"Recall         = {metrics['Recall']:.6f}"
        )

        print(
            f"F1             = {metrics['F1']:.6f}"
        )

        print(
            f"MCC            = {metrics['MCC']:.6f}"
        )

        print(
            f"Event Recall   = {event_metrics['Event_Recall']:.6f}"
        )

        print(
            f"Event Precision= {event_metrics['Event_Precision']:.6f}"
        )

        print(
            f"Median Lead    = {event_metrics['Median_First_Warning_Lead_Min']}"
        )

        print(
            f"FP origins/hr  = {event_metrics['False_Warning_Origins_Per_Hour']:.6f}"
        )


    del X_train_all_rep
    del X_val_all_rep
    del X_train
    del X_val

    gc.collect()


# ==================================================================================================
# V2 SUMMARY
# ==================================================================================================

results = pd.DataFrame(
    all_rows
)


results.to_csv(
    FINAL_DIR
    / "ALL_V2_VALIDATION_REPLICATES.csv",
    index=False
)


summary_rows = []


for H in HORIZONS:

    df = results.loc[
        results[
            "Horizon"
        ]
        ==
        H
    ]


    summary_rows.append(
        {
            "Horizon":
                H,

            "Representation":
                df[
                    "Representation"
                ].iloc[
                    0
                ],

            "Average_Precision_Mean":
                float(
                    df[
                        "Average_Precision"
                    ].mean()
                ),

            "Average_Precision_Std":
                float(
                    df[
                        "Average_Precision"
                    ].std(
                        ddof=1
                    )
                ),

            "PR_AUC_Mean":
                float(
                    df[
                        "PR_AUC"
                    ].mean()
                ),

            "ROC_AUC_Mean":
                float(
                    df[
                        "ROC_AUC"
                    ].mean()
                ),

            "Accuracy_Mean":
                float(
                    df[
                        "Accuracy"
                    ].mean()
                ),

            "Balanced_Accuracy_Mean":
                float(
                    df[
                        "Balanced_Accuracy"
                    ].mean()
                ),

            "Precision_Mean":
                float(
                    df[
                        "Precision"
                    ].mean()
                ),

            "Recall_Mean":
                float(
                    df[
                        "Recall"
                    ].mean()
                ),

            "F1_Mean":
                float(
                    df[
                        "F1"
                    ].mean()
                ),

            "MCC_Mean":
                float(
                    df[
                        "MCC"
                    ].mean()
                ),

            "Brier_Mean":
                float(
                    df[
                        "Brier"
                    ].mean()
                ),

            "Event_Recall_Mean":
                float(
                    df[
                        "Event_Recall"
                    ].mean()
                ),

            "Event_Precision_Mean":
                float(
                    df[
                        "Event_Precision"
                    ].mean()
                ),

            "Median_Lead_Mean":
                float(
                    df[
                        "Median_First_Warning_Lead_Min"
                    ].mean()
                ),

            "False_Warning_Origins_Per_Hour_Mean":
                float(
                    df[
                        "False_Warning_Origins_Per_Hour"
                    ].mean()
                ),
        }
    )


V2_SUMMARY = pd.DataFrame(
    summary_rows
)


V2_SUMMARY.to_csv(
    FINAL_DIR
    / "XGBOOST_V2_VALIDATION_SUMMARY.csv",
    index=False
)


print("\n" + "=" * 120)
print("XGBOOST V2 — OFFICIAL VALIDATION SUMMARY")
print("=" * 120)


print(
    V2_SUMMARY.to_string(
        index=False
    )
)


# ==================================================================================================
# COMPARE WITH XGBOOST V1
# ==================================================================================================

V1_SUMMARY_FILE = (
    V1_ROOT
    / "final_validation"
    / "XGBOOST_VALIDATION_SUMMARY.csv"
)


LSTM_SUMMARY_FILE = (
    ROOT
    / "models"
    / "LSTM_W10_PRIMARY"
    / "validation_diagnostics"
    / "LSTM_FULL_VALIDATION_SUMMARY.csv"
)


comparison_rows = []


V1 = (
    pd.read_csv(
        V1_SUMMARY_FILE
    )
    if V1_SUMMARY_FILE.is_file()
    else None
)


LSTM = (
    pd.read_csv(
        LSTM_SUMMARY_FILE
    )
    if LSTM_SUMMARY_FILE.is_file()
    else None
)


for H in HORIZONS:

    v2_row = V2_SUMMARY.loc[
        V2_SUMMARY[
            "Horizon"
        ]
        ==
        H
    ].iloc[
        0
    ]


    row = {
        "Horizon":
            H,

        "V2_Representation":
            v2_row[
                "Representation"
            ],

        "XGB_V2_AP":
            float(
                v2_row[
                    "Average_Precision_Mean"
                ]
            ),

        "XGB_V2_F1":
            float(
                v2_row[
                    "F1_Mean"
                ]
            ),

        "XGB_V2_MCC":
            float(
                v2_row[
                    "MCC_Mean"
                ]
            ),

        "XGB_V2_Event_Recall":
            float(
                v2_row[
                    "Event_Recall_Mean"
                ]
            ),
    }


    if V1 is not None:

        v1_row = V1.loc[
            V1[
                "Horizon"
            ]
            ==
            H
        ].iloc[
            0
        ]


        row[
            "XGB_V1_AP"
        ] = float(
            v1_row[
                "Average_Precision_Mean"
            ]
        )


        row[
            "V2_minus_V1_AP"
        ] = (
            row[
                "XGB_V2_AP"
            ]
            -
            row[
                "XGB_V1_AP"
            ]
        )


    if LSTM is not None:

        lstm_row = LSTM.loc[
            LSTM[
                "Horizon"
            ]
            ==
            H
        ].iloc[
            0
        ]


        row[
            "LSTM_AP"
        ] = float(
            lstm_row[
                "Average_Precision_Mean"
            ]
        )


        row[
            "V2_minus_LSTM_AP"
        ] = (
            row[
                "XGB_V2_AP"
            ]
            -
            row[
                "LSTM_AP"
            ]
        )


        row[
            "LSTM_F1"
        ] = float(
            lstm_row[
                "F1_Mean"
            ]
        )


        row[
            "V2_minus_LSTM_F1"
        ] = (
            row[
                "XGB_V2_F1"
            ]
            -
            row[
                "LSTM_F1"
            ]
        )


    comparison_rows.append(
        row
    )


comparison = pd.DataFrame(
    comparison_rows
)


comparison.to_csv(
    FINAL_DIR
    / "XGBOOST_V1_V2_LSTM_COMPARISON.csv",
    index=False
)


print("\n" + "=" * 120)
print("XGBOOST V1 vs XGBOOST V2 vs ORIGINAL LSTM")
print("=" * 120)


print(
    comparison.to_string(
        index=False
    )
)


# ==================================================================================================
# DEVELOPMENT LOCK
# ==================================================================================================

with open(
    V2_ROOT
    / "XGBOOST_V2_DEVELOPMENT_LOCK.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        {
            "Model":
                "XGBOOST_W10_TEMPORAL_V2",

            "Status":
                "TRAIN_VALIDATION_DEVELOPMENT_COMPLETE",

            "Representation_Search":
                [
                    "SUMMARY98",
                    "FLAT140",
                    "HYBRID238",
                    "HYBRID_DIFF364",
                ],

            "Representation_Selected_Using":
                "TRAIN_INTERNAL_EPISODE_CV",

            "Hyperparameters_Selected_Using":
                "TRAIN_INTERNAL_EPISODE_CV",

            "Official_Validation_Used_For_HPO":
                False,

            "Official_Validation_Used_For_Final_Confirmation":
                True,

            "Final_Seeds":
                FINAL_SEEDS,

            "TEST_Path_Defined":
                False,

            "TEST_Loaded":
                False,

            "TEST_Used":
                False,

            "TEST_Evaluation_Performed":
                False,
        },
        f,
        indent=4
    )


print("\n" + "=" * 120)
print("✓ XGBOOST V2 DEVELOPMENT COMPLETE")
print("=" * 120)


print(
    "V1 preserved ✓"
)

print(
    "Representation optimized ✓"
)

print(
    "Exact temporal ordering tested ✓"
)

print(
    "First differences tested ✓"
)

print(
    "Hyperparameters seriously optimized ✓"
)

print(
    "Official validation used only after TRAIN-CV selection ✓"
)

print(
    "TEST path defined: NO ✓"
)

print(
    "TEST loaded: NO ✓"
)

print(
    "TEST evaluated: NO ✓"
)

print(
    "\nDO NOT RUN TEST."
)

print("=" * 120)

gc.collect()